# QM9 `gap_eV` 予測 — 最終版（3方針の正式実装 + 交差検証・再現性の修正）

配布された `smiles` だけから特徴量を生成し、`gap_eV`（HOMO-LUMO ギャップ, eV）を予測する。
評価指標は **MAE**。全方針で **同一の 5-fold グループ交差検証**（canonical SMILES をグループにした
`GroupKFold(n_splits=5, shuffle=True, random_state=8)`）を共有し、重複分子のfold跨ぎリークを防いで公平に比較する。

## 当初決定した3方針（変更せず正式実装する）

| 方針 | 特徴量 | モデル | リーク対策の要点 |
| --- | --- | --- | --- |
| **方針1** | 重要RDKit特徴量（標準2D記述子＋独自特徴量） | LightGBM / XGBoost | 特徴量選択を**各学習fold内**で実行 |
| **方針2** | RDKit特徴量＋PCA | RBF-SVR | `SimpleImputer→StandardScaler→PCA→SVR` を **Pipeline** 化（各fold内fit） |
| **方針3** | 大域RDKit＋独自特徴量＋Morgan FP | LightGBM / XGBoost | 特徴量設定を**モデル別**に選択、列サンプリング＋正則化で過学習抑制 |

## この最終版で入れた修正（`qm9_gap_prediction_revised.ipynb` からの差分）
1. **最終実行は `QUICK=False`（FULL）**。QUICKモードは開発用に残す。Optuna試行数は変数化。
2. **重複分子を同じfoldへ**：canonical SMILES をグループにした `GroupKFold`。fold割当を CSV/NumPy で保存。
3. **CV前の全データ中央値補完を削除**：方針1/3 は NaN をそのままLGBM/XGBへ、方針2は Pipeline 内で補完。
4. **early stopping 後に学習fold全体で再学習**（`fit_boost_with_refit`）。LGBは `best_iteration_`、XGBは `best_iteration+1`。
5. **Morganキャッシュに SMILES の SHA-1** など内容ハッシュを追加（行数一致でも中身が違えば再計算）。
6. **方針3の特徴量設定をLightGBM/XGBoostで別々に選択**。`fr_*` は除外/残すを比較。
7. **旧 MAE≈0.2098 モデルを再現**（`baseline_legacy_rdkit_morgan_lgbm`）。厳密再現と共通fold上の誠実な再評価を両方出す。
8. **方針2のPCA設定を全候補比較**（粗探索→上位2設定をOptuna詳細探索）。FULLはSVRをサブサンプルしない。
9. **SHAPサンプルをseed固定のランダム抽出**に変更。
10. **Optuna全trial履歴をCSV保存＋SQLite storageで再開可能**に。
11. **最終提出候補はCV MAEとモデル多様性で決定**（旧ベースライン・RF・ブレンドは参考候補として比較）。

## コンペ制約（順守事項）
- 外部データを追加しない（配布SMILESからの特徴量生成のみ）。乱数シードは `8` に統一。
- テストの並び順・予測値は手作業で変更しない。提出は最大3モデル、各CSVは `smiles,gap_eV` の列順・4000行。

## 出力
`submission_strategy1.csv` / `submission_strategy2.csv` / `submission_strategy3.csv`（＋参考: ブレンド）

## 2. ライブラリ・設定

**何をする処理か**：必要ライブラリの読み込み、全乱数シードの固定（`8`）、実行モード（`QUICK`/`FULL`）と
探索設定・Optuna試行数・入出力パスの一元定義。

**なぜ必要か**：再現性の担保と、環境に応じた探索規模の切り替えのため。まず `QUICK=True` で最後まで通し、
値を確認してから `QUICK=False` でフル探索する運用を想定する。**最終実行は `QUICK=False`**。

In [1]:
import os, sys, json, time, random, hashlib, platform, warnings
from pathlib import Path
from typing import Callable, Dict, List, Sequence, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats as ss
from scipy.optimize import minimize

import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.rdchem import HybridizationType, BondType

import sklearn
from sklearn.model_selection import KFold, GroupKFold, train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

import lightgbm as lgb
import xgboost as xgb
import optuna
import shap
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)

LIB_VERSIONS = {
    "python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine(),
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": __import__("scipy").__version__,
    "scikit-learn": sklearn.__version__, "lightgbm": lgb.__version__, "xgboost": xgb.__version__,
    "optuna": optuna.__version__, "shap": shap.__version__, "rdkit": rdkit.__version__,
}
for k, v in LIB_VERSIONS.items():
    print(f"{k:14s}: {v}")

/Users/fu-riku/Library/CloudStorage/GoogleDrive-fukumoto.riku.fr7@g.ext.naist.jp/マイドライブ/MI_Lab_cloud/python-seminar/lesson_9/self-code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python        : 3.11.15
platform      : macOS-14.5-arm64-arm-64bit
machine       : arm64
numpy         : 2.4.6
pandas        : 3.0.3
scipy         : 1.17.1
scikit-learn  : 1.9.0
lightgbm      : 4.7.0
xgboost       : 3.2.0
optuna        : 4.9.0
shap          : 0.51.0
rdkit         : 2026.03.4


### 設定（シード・実行モード・探索規模・Optuna試行数）

- `QUICK=True`：Optuna試行数・設定スイープ・SVR学習件数を縮小した**動作確認用**。
- `QUICK=False`：仕様どおりの**フル探索**（数時間規模になり得る）。**最終実行はこちら**。

FULL探索の最低条件（仕様）：方針1 LightGBM/XGBoost 各30 trial以上、方針2 40 trial以上、
方針3 LightGBM/XGBoost 各30 trial以上、特徴量設定スイープは 5-fold すべてを使用。

Optunaの試行数は下の `N_TRIALS_*` 変数で変更できる。CPUのみ・GPU不要で動作する。

In [2]:
SEED = 8
N_SPLITS = 5
QUICK = False   # ← 動作確認は True、最終フル探索は False（本ノートブックの最終実行は False）

def set_all_seeds(seed: int) -> None:
    """Python / NumPy / ハッシュシードをまとめて固定する。"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(SEED)
RUN_MODE = "QUICK" if QUICK else "FULL"

# --- Optuna 試行数（変数で変更可能。FULLは仕様の最低条件を満たす）---
N_TRIALS_S1_LGB = 6 if QUICK else 30
N_TRIALS_S1_XGB = 6 if QUICK else 30
N_TRIALS_S2_SVR = 8 if QUICK else 40
N_TRIALS_S3_LGB = 6 if QUICK else 30
N_TRIALS_S3_XGB = 6 if QUICK else 30

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "qm9_bandgap_train.csv"
TEST_PATH = DATA_DIR / "qm9_bandgap_test_without_answer.csv"
CACHE_DIR = Path("cache");    CACHE_DIR.mkdir(exist_ok=True)      # 特徴量キャッシュ
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)   # OOF/テスト予測・結果JSON・trial履歴
OPTUNA_STORAGE = f"sqlite:///{(RESULTS_DIR / 'optuna.db').as_posix()}"  # study再開用

# 探索規模（QUICK/FULL 切り替え）
CFG: Dict[str, object] = dict(
    boost_lr=0.05,
    boost_max_estimators=(500 if QUICK else 2000),            # baseline/スイープ/ES上限の木数
    boost_nest_range=((200, 700) if QUICK else (400, 3000)),  # Optunaの n_estimators 探索範囲（ES上限）
    early_stopping_rounds=(50 if QUICK else 100),
    # 方針1
    s1_feature_counts=([20, 80, "all"] if QUICK else [10, 20, 40, 80, 120, "all"]),
    s1_perm_repeats=(2 if QUICK else 5),
    s1_shap_sample=(400 if QUICK else 2000),
    # 方針2
    s2_pca_var=([0.95, 0.99] if QUICK else [0.90, 0.95, 0.97, 0.99]),
    s2_pca_ncomp=([40, 120] if QUICK else [20, 40, 80, 120]),
    s2_coarse_svr=dict(C=10.0, gamma="scale", epsilon=0.1),   # 粗探索の固定SVR
    s2_top_pca=2,                                             # 詳細探索する上位PCA設定数
    s2_svr_max_train=(2000 if QUICK else None),              # QUICKのみサブサンプル、FULLは全件
    # 方針3
    s3_corr_thresholds=([0.98, None] if QUICK else [0.95, 0.98, 0.995, None]),  # None=高相関列を残す
    s3_fr_options=([False] if QUICK else [False, True]),      # fr_* を除外(False)/残す(True)
    s3_morgan_variants=([("count", 2, 2048)] if QUICK
                        else [("count", 2, 2048), ("binary", 2, 2048), ("count", 3, 2048)]),
    s3_lowfreq_min_df=([1, 5] if QUICK else [1, 2, 5, 10]),   # 出現分子数の下限（1=未出現のみ削除）
    sweep_n_splits=(2 if QUICK else N_SPLITS),                # 特徴量設定スイープに使うfold数（FULLは5-fold）
)
print("run_mode =", RUN_MODE, "| QUICK =", QUICK)
print("Optuna trials: s1_lgb=%d s1_xgb=%d s2_svr=%d s3_lgb=%d s3_xgb=%d"
      % (N_TRIALS_S1_LGB, N_TRIALS_S1_XGB, N_TRIALS_S2_SVR, N_TRIALS_S3_LGB, N_TRIALS_S3_XGB))
print(json.dumps({k: v for k, v in CFG.items()}, ensure_ascii=False, indent=2, default=str))

run_mode = FULL | QUICK = False
Optuna trials: s1_lgb=30 s1_xgb=30 s2_svr=40 s3_lgb=30 s3_xgb=30
{
  "boost_lr": 0.05,
  "boost_max_estimators": 2000,
  "boost_nest_range": [
    400,
    3000
  ],
  "early_stopping_rounds": 100,
  "s1_feature_counts": [
    10,
    20,
    40,
    80,
    120,
    "all"
  ],
  "s1_perm_repeats": 5,
  "s1_shap_sample": 2000,
  "s2_pca_var": [
    0.9,
    0.95,
    0.97,
    0.99
  ],
  "s2_pca_ncomp": [
    20,
    40,
    80,
    120
  ],
  "s2_coarse_svr": {
    "C": 10.0,
    "gamma": "scale",
    "epsilon": 0.1
  },
  "s2_top_pca": 2,
  "s2_svr_max_train": null,
  "s3_corr_thresholds": [
    0.95,
    0.98,
    0.995,
    null
  ],
  "s3_fr_options": [
    false,
    true
  ],
  "s3_morgan_variants": [
    [
      "count",
      2,
      2048
    ],
    [
      "binary",
      2,
      2048
    ],
    [
      "count",
      3,
      2048
    ]
  ],
  "s3_lowfreq_min_df": [
    1,
    2,
    5,
    10
  ],
  "sweep_n_splits": 5
}


### 共通ユーティリティ

**何をする処理か**：全方針で共有する関数（fold反復・ブースティングの2段階学習/予測・結果保存・提出CSV検証）を定義する。

**なぜ必要か**：全方針で**同一fold**・同一の早期終了条件・同一の検証手順を使い、リークと不整合を防ぐため。

**修正4（再学習）**：`fit_boost_with_refit()` は (1) 学習foldの内部85/15分割で early stopping して `best_iteration` を得て、
(2) 学習fold**全体**を `n_estimators=best_iteration`（XGBは `+1`）で **early stopping 無し**に再学習する。
外側検証foldは一切触らない。LightGBMは `best_iteration_`、XGBoostは `best_iteration+1` を採用木数にする。

In [3]:
RESULTS: List[dict] = []   # §14 の比較表に集約する行

def iter_folds(fold_id: np.ndarray, n_splits: int):
    """fold_id 配列から (train_idx, valid_idx) を順に返す。全方針で共有。"""
    idx = np.arange(len(fold_id))
    for f in range(n_splits):
        yield idx[fold_id != f], idx[fold_id == f]

def _lgb_params(params: Optional[dict], seed: int) -> dict:
    base = dict(objective="regression_l1", metric="mae",
                n_estimators=CFG["boost_max_estimators"], learning_rate=CFG["boost_lr"],
                random_state=seed, n_jobs=-1, verbose=-1)
    if params:
        base.update(params)
    return base

def _xgb_params(params: Optional[dict], seed: int) -> dict:
    base = dict(objective="reg:absoluteerror", eval_metric="mae",
                n_estimators=CFG["boost_max_estimators"], learning_rate=CFG["boost_lr"],
                random_state=seed, n_jobs=-1, tree_method="hist", missing=np.nan)
    if params:
        base.update(params)
    return base

def make_lgb(params: Optional[dict] = None, seed: int = SEED) -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(**_lgb_params(params, seed))

def make_xgb(params: Optional[dict] = None, seed: int = SEED) -> xgb.XGBRegressor:
    return xgb.XGBRegressor(**_xgb_params(params, seed),
                            early_stopping_rounds=CFG["early_stopping_rounds"])

def fit_boost_with_refit(model_kind: str, params: Optional[dict], X_train, y_train, seed: int):
    """修正4：2段階学習。
    第1段階：学習fold内部を85/15分割し early stopping で best_iteration を決める（外側検証foldは不使用）。
    第2段階：学習fold全体を n_estimators=採用木数・early stopping無しで再学習する。
    戻り値: (再学習済みモデル, 採用したn_estimators, 内部early stoppingのbest_iteration)。"""
    params = dict(params or {})
    max_est = int(params.get("n_estimators", CFG["boost_max_estimators"]))  # ES段の木数上限
    xi, xv, yi, yv = train_test_split(X_train, y_train, test_size=0.15, random_state=seed)
    if model_kind == "lgb":
        es = lgb.LGBMRegressor(**_lgb_params({**params, "n_estimators": max_est}, seed))
        es.fit(xi, yi, eval_set=[(xv, yv)], eval_metric="mae",
               callbacks=[lgb.early_stopping(CFG["early_stopping_rounds"], verbose=False),
                          lgb.log_evaluation(0)])
        best_it = int(es.best_iteration_ or max_est)
        n_adopt = max(best_it, 1)                       # LightGBM: n_estimators = best_iteration_
        final = lgb.LGBMRegressor(**_lgb_params({**params, "n_estimators": n_adopt}, seed))
        final.fit(X_train, y_train)                     # early stopping無し・eval_set無し
    elif model_kind == "xgb":
        es = xgb.XGBRegressor(**_xgb_params({**params, "n_estimators": max_est}, seed),
                              early_stopping_rounds=CFG["early_stopping_rounds"])
        es.fit(xi, yi, eval_set=[(xv, yv)], verbose=False)
        bi = es.best_iteration
        best_it = int(bi if bi is not None else max_est - 1)
        n_adopt = best_it + 1                           # XGBoost: n_estimators = best_iteration + 1
        final = xgb.XGBRegressor(**_xgb_params({**params, "n_estimators": n_adopt}, seed))
        final.fit(X_train, y_train)                     # early stopping無し・eval_set無し
    else:
        raise ValueError(f"unknown model_kind: {model_kind}")
    return final, n_adopt, best_it

def cv_boost(get_fold_data, y, fold_id, kind: str, params: Optional[dict], seed: int = SEED):
    """共有foldでブースティングをCV（各foldで fit_boost_with_refit）。
    get_fold_data(f, tr, va) -> (X_tr_sub, X_va_sub, X_te)（行スライス済み。列はfoldごとに異なってよい）。
    戻り値: oof, test_pred(fold平均), fold_mae, adopted_estimators, best_iters。"""
    oof = np.zeros(len(y))
    test_pred = None
    fold_mae, adopted, best_iters = [], [], []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        X_tr_sub, X_va_sub, X_te = get_fold_data(f, tr, va)
        model, n_ad, bi = fit_boost_with_refit(kind, params, X_tr_sub, y[tr], seed + f)
        oof[va] = model.predict(X_va_sub)
        te = model.predict(X_te)
        if test_pred is None:
            test_pred = np.zeros(len(te))
        test_pred += te / N_SPLITS
        fold_mae.append(mean_absolute_error(y[va], oof[va]))
        adopted.append(int(n_ad)); best_iters.append(int(bi))
    return oof, test_pred, fold_mae, adopted, best_iters

def register_result(strategy, feature_set, model, n_features, fold_mae, train_time,
                    best_params, submission_path, oof=None, test_pred=None, extra=None):
    """結果を RESULTS に追加し、OOF/テスト予測と要約JSONを results/ に保存する。"""
    fold_mae = list(map(float, fold_mae))
    try:
        n_features = int(n_features)
    except (TypeError, ValueError):
        pass
    row = dict(strategy=strategy, feature_set=feature_set, model=model, n_features=n_features,
               run_mode=RUN_MODE, quick=QUICK,
               cv_mae_mean=float(np.mean(fold_mae)), cv_mae_std=float(np.std(fold_mae)),
               **{f"fold{i+1}_mae": v for i, v in enumerate(fold_mae)},
               training_time=float(train_time), best_params=json.dumps(best_params, default=str),
               submission_path=submission_path)
    if extra:
        row.update(extra)
    RESULTS.append(row)
    tag = strategy.replace(" ", "_").replace("/", "_")
    json.dump({**row, "lib_versions": LIB_VERSIONS}, open(RESULTS_DIR / f"{tag}.json", "w"),
              ensure_ascii=False, indent=2)
    if oof is not None:
        np.save(RESULTS_DIR / f"{tag}_oof.npy", np.asarray(oof))
    if test_pred is not None:
        np.save(RESULTS_DIR / f"{tag}_test.npy", np.asarray(test_pred))
    print(f"[登録] {strategy}: CV MAE = {row['cv_mae_mean']:.4f} ± {row['cv_mae_std']:.4f}  (run_mode={RUN_MODE})")
    return row

def make_submission(pred: np.ndarray, filename: str, test_df: pd.DataFrame) -> pd.DataFrame:
    """test と同じ順序・行数で smiles,gap_eV のCSVを書き出し、妥当性を検証する。"""
    sub = pd.DataFrame({"smiles": test_df["smiles"].to_numpy(),
                        "gap_eV": np.asarray(pred, dtype=np.float64)})
    assert len(sub) == len(test_df), "行数がテストと不一致"
    assert list(sub.columns) == ["smiles", "gap_eV"], "列名/列順が不正"
    assert sub["smiles"].tolist() == test_df["smiles"].tolist(), "SMILES順序がテストと不一致"
    assert sub["gap_eV"].notna().all(), "欠損予測あり"
    assert np.isfinite(sub["gap_eV"].to_numpy()).all(), "無限値あり"
    sub.to_csv(filename, index=False)
    print(f"保存: {filename}  ({len(sub)}行)")
    return sub

## 3. データ読み込み

**何をする処理か**：学習データ（`smiles`,`gap_eV`）とテストデータ（`smiles`）を読み込み、形状と目的変数分布を確認する。

**なぜ必要か**：以降の全処理の入力。列名・行数が想定どおりか（train 15000 / test 4000）を最初に確認する。

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("train:", train_df.shape, " test:", test_df.shape)
assert list(train_df.columns) == ["smiles", "gap_eV"], "train の列が想定と異なる"
assert list(test_df.columns) == ["smiles"], "test の列が想定と異なる"
y = train_df["gap_eV"].to_numpy(dtype=np.float64)
display(train_df.head(3))
print(train_df["gap_eV"].describe())

train: (15000, 2)  test: (4000, 1)


,smiles,gap_eV
0,C#CC1(CNC1=O)C#C,7.094012
1,CC1CC=CCOC=N1,6.854552
2,CC1=C2CCC3C(C1)C23,5.839566


count    15000.000000
mean         6.823883
std          1.288640
min          1.774183
25%          5.885826
50%          6.781081
75%          7.834162
max         10.718570
Name: gap_eV, dtype: float64


## 4. データ監査

**何をする処理か**：SMILES欠損 / RDKit変換不能 / 重複 / canonical重複 / 同一canonicalに異なる `gap_eV` /
目的変数の欠損・分布 / train・testの元素・サイズ分布差を確認する。

**なぜ必要か**：リークやデータ不整合を早期に発見するため。**無効SMILESは削除せず記録**する
（テスト側に変換不能があると4000行の予測が作れないため、その場合のみ停止）。
canonical SMILES の重複は **修正2（GroupKFold）** で「同一分子を同じfoldへ」入れる根拠になる。

In [5]:
def smiles_to_mols(smiles: Sequence[str]) -> Tuple[List, List[Tuple[int, str]]]:
    """SMILES列を Mol へ変換し (mols, [(行番号, SMILES)] の無効リスト) を返す。行は削除しない。"""
    mols, invalid = [], []
    for i, smi in enumerate(smiles):
        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
        if m is None:
            invalid.append((i, smi))
        mols.append(m)
    return mols, invalid

train_mols, train_invalid = smiles_to_mols(train_df["smiles"].tolist())
test_mols, test_invalid = smiles_to_mols(test_df["smiles"].tolist())

audit = {}
audit["train_smiles_missing"] = int(train_df["smiles"].isna().sum())
audit["test_smiles_missing"] = int(test_df["smiles"].isna().sum())
audit["train_invalid_mol"] = len(train_invalid)
audit["test_invalid_mol"] = len(test_invalid)
audit["train_dup_smiles"] = int(train_df["smiles"].duplicated().sum())
audit["test_dup_smiles"] = int(test_df["smiles"].duplicated().sum())
audit["gap_missing"] = int(train_df["gap_eV"].isna().sum())

canon = [Chem.MolToSmiles(m) if m is not None else None for m in train_mols]
train_df = train_df.assign(_canon=canon)
audit["train_dup_canonical"] = int(train_df["_canon"].duplicated().sum())
gap_by_canon = train_df.dropna(subset=["_canon"]).groupby("_canon")["gap_eV"].nunique()
audit["canonical_gap_conflicts"] = int((gap_by_canon > 1).sum())

print(json.dumps(audit, ensure_ascii=False, indent=2))
if train_invalid or test_invalid:
    print("\n[無効SMILES 記録（削除はしない）]")
    for idx, smi in (train_invalid + test_invalid)[:20]:
        print("  row", idx, repr(smi))
assert not test_invalid, "テストに変換不能なSMILESがあります（4000行の予測が作れません）"

{
  "train_smiles_missing": 0,
  "test_smiles_missing": 0,
  "train_invalid_mol": 0,
  "test_invalid_mol": 0,
  "train_dup_smiles": 2,
  "test_dup_smiles": 0,
  "gap_missing": 0,
  "train_dup_canonical": 2,
  "canonical_gap_conflicts": 0
}


In [6]:
# 目的変数分布 / train-test の元素・サイズ分布差（監査の数値サマリ）
n_heavy_tr = np.array([m.GetNumHeavyAtoms() if m else 0 for m in train_mols])
n_heavy_te = np.array([m.GetNumHeavyAtoms() if m else 0 for m in test_mols])
print("重原子数  train: mean=%.2f min=%d max=%d | test: mean=%.2f min=%d max=%d"
      % (n_heavy_tr.mean(), n_heavy_tr.min(), n_heavy_tr.max(),
         n_heavy_te.mean(), n_heavy_te.min(), n_heavy_te.max()))

def element_composition(mols):
    from collections import Counter
    c = Counter()
    for m in mols:
        if m is None:
            continue
        for a in m.GetAtoms():
            if a.GetAtomicNum() > 1:
                c[a.GetSymbol()] += 1
    tot = sum(c.values()) or 1
    return {k: v / tot for k, v in sorted(c.items())}

comp = pd.DataFrame({"train_frac": element_composition(train_mols),
                     "test_frac": element_composition(test_mols)}).fillna(0.0)
print("\n元素構成（重原子ベースの割合）:")
display(comp)
print("gap_eV: mean=%.3f std=%.3f min=%.3f max=%.3f" % (y.mean(), y.std(), y.min(), y.max()))

重原子数  train: mean=8.80 min=6 max=9 | test: mean=8.92 min=8 max=9

元素構成（重原子ベースの割合）:


,train_frac,test_frac
C,0.720207,0.736672
F,0.001704,0.000000
N,0.118256,0.099165
O,0.159833,0.164163


gap_eV: mean=6.824 std=1.289 min=1.774 max=10.719


## 5. 共通fold作成（修正2：GroupKFold で重複分子を同じfoldへ）

**何をする処理か**：canonical SMILES を**グループ**として `GroupKFold(n_splits=5, shuffle=True, random_state=8)`
で各サンプルにfold番号を割り当てる。RDKit変換不能な行は元のSMILES文字列をグループ代替に使う。

**なぜ必要か**：学習データには canonical SMILES の重複がある（§4監査）。通常の `KFold` では同一分子が
学習foldと検証foldへ分かれ、CV MAE が楽観側にリークする。同一分子を必ず同じfoldに入れて防ぐ。

**確認事項**：(a) 同一canonicalが複数foldへ分割されていない、(b) 全サンプルが一度だけ検証foldへ入る、
(c) 3方針で同じfold（`fold_id`）を使う、(d) fold割当を CSV と NumPy で保存する。

In [7]:
# canonical SMILES をグループにする（変換不能行は元SMILESで代替）
smiles_arr = train_df["smiles"].astype(str).to_numpy()
groups = np.array([g if isinstance(g, str) else smiles_arr[i]
                   for i, g in enumerate(train_df["_canon"].to_numpy())])

gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_id = np.full(len(train_df), -1, dtype=int)
for f, (_, va) in enumerate(gkf.split(train_df, y, groups)):
    fold_id[va] = f
assert (fold_id >= 0).all(), "fold未割当の行がある"

# 検証：全サンプルが一度だけ検証foldへ / 同一グループが単一foldか
assert int(np.bincount(fold_id).sum()) == len(train_df), "検証fold件数の合計がサンプル数と不一致"
span = pd.DataFrame({"group": groups, "fold": fold_id}).groupby("group")["fold"].nunique()
assert (span == 1).all(), "同一canonical SMILESが複数foldにまたがっている"

# 保存（CSV + NumPy）
np.save(RESULTS_DIR / "fold_id.npy", fold_id)
pd.DataFrame({"index": np.arange(len(fold_id)), "smiles": smiles_arr,
              "canonical_smiles": groups, "fold": fold_id}
             ).to_csv(RESULTS_DIR / "fold_assignment.csv", index=False)
print("fold サイズ:", np.bincount(fold_id))
print("ユニークgroup数:", int(len(span)), "| 全groupが単一fold:", bool((span == 1).all()))
print("同一canonicalがfoldをまたがない:", bool((span == 1).all()))

fold サイズ: [3000 3001 3000 2999 3000]
ユニークgroup数: 14998 | 全groupが単一fold: True
同一canonicalがfoldをまたがない: True


## 6. RDKit標準2D記述子生成

**何をする処理か**：`Descriptors.CalcMolDescriptors()` で標準2D記述子（約200個）を一括計算する。3D記述子は使わない。

**なぜ必要か**：分子量・環数・極性表面積など、化学的に解釈しやすい大域特徴量を得るため。記述子名一覧とRDKitバージョンはキャッシュに保存する。

In [8]:
def calc_descriptors(mols) -> pd.DataFrame:
    """RDKit標準2D記述子を一括計算して DataFrame で返す（None mol は空行→後段でNaN）。"""
    rows = [({} if m is None else Descriptors.CalcMolDescriptors(m))
            for m in tqdm(mols, desc="descriptors")]
    return pd.DataFrame(rows)

def file_signature(path: Path) -> dict:
    return {"name": Path(path).name, "sha1": hashlib.sha1(Path(path).read_bytes()).hexdigest()}

def cached_frame(name: str, compute: Callable[[], pd.DataFrame], meta: dict) -> pd.DataFrame:
    """設定(meta)が一致すればParquetキャッシュを読み、変われば再計算して保存する。"""
    pq, mj = CACHE_DIR / f"{name}.parquet", CACHE_DIR / f"{name}.meta.json"
    if pq.exists() and mj.exists() and json.load(open(mj)) == meta:
        print(f"[cache hit] {name}")
        return pd.read_parquet(pq)
    df = compute()
    df.to_parquet(pq)
    json.dump(meta, open(mj, "w"), ensure_ascii=False, indent=2)
    print(f"[cache save] {name}  shape={df.shape}")
    return df

_desc_meta = dict(kind="rdkit_descriptors", rdkit=rdkit.__version__, src=file_signature(TRAIN_PATH))
desc_train = cached_frame("desc_train", lambda: calc_descriptors(train_mols), _desc_meta)
desc_test = cached_frame("desc_test", lambda: calc_descriptors(test_mols),
                         dict(_desc_meta, src=file_signature(TEST_PATH)))
DESC_NAMES = list(desc_train.columns)
json.dump({"rdkit": rdkit.__version__, "descriptor_names": DESC_NAMES},
          open(RESULTS_DIR / "descriptor_names.json", "w"), ensure_ascii=False, indent=2)
print("記述子数:", desc_train.shape[1])

[cache hit] desc_train
[cache hit] desc_test
記述子数: 217


## 7. 独自特徴量生成

**何をする処理か**：`Mol`/`Atom`/`Bond`/`RingInfo` から、元素・組成 / 原子状態 / 結合 / 環 / 比率の独自特徴量を生成する。
元素は train・test に実在する重元素を走査して列を作る（QM9 では C/N/O/F）。比率はゼロ除算を防止する。

**なぜ必要か**：標準記述子が直接持たない、組成・混成・環種・形式電荷などの明示的な数え上げを補うため。

**リーク上の注意**：目的変数を使わない純粋な構造特徴。列そのものにリークはない。

In [9]:
def get_heavy_element_set(*mol_lists) -> List[str]:
    """train・test に実在する重元素（原子番号>1）の記号集合を返す。"""
    syms = set()
    for mols in mol_lists:
        for m in mols:
            if m is None:
                continue
            for a in m.GetAtoms():
                if a.GetAtomicNum() > 1:
                    syms.add(a.GetSymbol())
    return sorted(syms)

def _safe_div(a: float, b: float) -> float:
    return float(a) / float(b) if b else 0.0

def calc_custom_features(m, elements: Sequence[str]) -> Dict[str, float]:
    """1分子の独自特徴量を dict で返す（None mol は空 dict）。"""
    if m is None:
        return {}
    atoms, bonds, ri = list(m.GetAtoms()), list(m.GetBonds()), m.GetRingInfo()
    n_heavy = m.GetNumHeavyAtoms()
    n_H = sum(a.GetTotalNumHs() for a in atoms) + sum(1 for a in atoms if a.GetAtomicNum() == 1)
    n_total = n_heavy + n_H
    elem_counts = {f"cnt_{s}": 0 for s in elements}
    n_hetero = n_aromatic_atom = n_ring_atom = n_sp = n_sp2 = n_sp3 = 0
    deg = {1: 0, 2: 0, 3: 0, 4: 0}
    fc_sum = fc_abs = n_pos = n_neg = 0
    for a in atoms:
        if a.GetAtomicNum() <= 1:
            continue
        s = a.GetSymbol()
        if f"cnt_{s}" in elem_counts:
            elem_counts[f"cnt_{s}"] += 1
        if s != "C":
            n_hetero += 1
        n_aromatic_atom += int(a.GetIsAromatic())
        n_ring_atom += int(a.IsInRing())
        hyb = a.GetHybridization()
        n_sp += int(hyb == HybridizationType.SP)
        n_sp2 += int(hyb == HybridizationType.SP2)
        n_sp3 += int(hyb == HybridizationType.SP3)
        d = a.GetDegree()
        if d in deg:
            deg[d] += 1
        fc = a.GetFormalCharge()
        fc_sum += fc; fc_abs += abs(fc)
        n_pos += int(fc > 0); n_neg += int(fc < 0)

    n_bonds = len(bonds)
    n_single = n_double = n_triple = n_arom_bond = n_conj = n_ring_bond = 0
    for b in bonds:
        bt = b.GetBondType()
        n_single += int(bt == BondType.SINGLE)
        n_double += int(bt == BondType.DOUBLE)
        n_triple += int(bt == BondType.TRIPLE)
        n_arom_bond += int(bt == BondType.AROMATIC or b.GetIsAromatic())
        n_conj += int(b.GetIsConjugated())
        n_ring_bond += int(b.IsInRing())

    ring_sizes = [len(r) for r in ri.AtomRings()]
    n_rings = len(ring_sizes)
    ring_size_counts = {f"ring_{k}": sum(1 for z in ring_sizes if z == k) for k in range(3, 9)}
    max_ring = max(ring_sizes) if ring_sizes else 0
    n_arom_ring = rdMolDescriptors.CalcNumAromaticRings(m)
    n_aliph_ring = rdMolDescriptors.CalcNumAliphaticRings(m)
    n_hetero_ring = (rdMolDescriptors.CalcNumAromaticHeterocycles(m)
                     + rdMolDescriptors.CalcNumAliphaticHeterocycles(m))
    n_carbo_ring = (rdMolDescriptors.CalcNumAromaticCarbocycles(m)
                    + rdMolDescriptors.CalcNumAliphaticCarbocycles(m))
    n_spiro = rdMolDescriptors.CalcNumSpiroAtoms(m)
    n_bridge = rdMolDescriptors.CalcNumBridgeheadAtoms(m)
    n_multiple = n_double + n_triple + n_arom_bond

    feat: Dict[str, float] = {k: float(v) for k, v in elem_counts.items()}
    feat.update(dict(
        cnt_H=float(n_H), n_heavy=float(n_heavy), n_total_atoms=float(n_total),
        n_hetero=float(n_hetero), fc_sum=float(fc_sum), fc_abs_sum=float(fc_abs),
        n_aromatic_atom=float(n_aromatic_atom), n_ring_atom=float(n_ring_atom),
        n_sp=float(n_sp), n_sp2=float(n_sp2), n_sp3=float(n_sp3),
        n_deg1=float(deg[1]), n_deg2=float(deg[2]), n_deg3=float(deg[3]), n_deg4=float(deg[4]),
        n_pos_charge=float(n_pos), n_neg_charge=float(n_neg),
        n_bonds=float(n_bonds), n_single=float(n_single), n_double=float(n_double),
        n_triple=float(n_triple), n_arom_bond=float(n_arom_bond), n_conj_bond=float(n_conj),
        n_ring_bond=float(n_ring_bond),
        n_rings=float(n_rings), max_ring_size=float(max_ring),
        n_aromatic_ring=float(n_arom_ring), n_aliphatic_ring=float(n_aliph_ring),
        n_hetero_ring=float(n_hetero_ring), n_carbo_ring=float(n_carbo_ring),
        n_spiro=float(n_spiro), n_bridgehead=float(n_bridge),
    ))
    feat.update({k: float(v) for k, v in ring_size_counts.items()})
    feat.update(dict(
        r_hetero=_safe_div(n_hetero, n_heavy), r_aromatic_atom=_safe_div(n_aromatic_atom, n_heavy),
        r_ring_atom=_safe_div(n_ring_atom, n_heavy), r_sp=_safe_div(n_sp, n_heavy),
        r_sp2=_safe_div(n_sp2, n_heavy), r_sp3=_safe_div(n_sp3, n_heavy),
        r_double=_safe_div(n_double, n_bonds), r_triple=_safe_div(n_triple, n_bonds),
        r_arom_bond=_safe_div(n_arom_bond, n_bonds), r_conj_bond=_safe_div(n_conj, n_bonds),
        r_multiple=_safe_div(n_multiple, n_bonds),
    ))
    return feat

def calc_custom_frame(mols, elements) -> pd.DataFrame:
    return pd.DataFrame([calc_custom_features(m, elements) for m in tqdm(mols, desc="custom")])

ELEMENTS = get_heavy_element_set(train_mols, test_mols)
print("検出した重元素:", ELEMENTS)
_cust_meta = dict(kind="custom", rdkit=rdkit.__version__, elements=ELEMENTS, src=file_signature(TRAIN_PATH))
cust_train = cached_frame("cust_train", lambda: calc_custom_frame(train_mols, ELEMENTS), _cust_meta)
cust_test = cached_frame("cust_test", lambda: calc_custom_frame(test_mols, ELEMENTS),
                         dict(_cust_meta, src=file_signature(TEST_PATH)))
print("独自特徴量数:", cust_train.shape[1])

検出した重元素: ['C', 'F', 'N', 'O']
[cache hit] cust_train
[cache hit] cust_test
独自特徴量数: 53


## 8. Morgan fingerprint 生成（修正5：キャッシュに内容ハッシュを付与）

**何をする処理か**：`rdFingerprintGenerator.GetMorganGenerator()` で Morgan FP を生成する。第一候補は
**count / radius=2 / fpSize=2048 / includeChirality=False**。方針3では count・binary、radius=2・3 も比較する。

**なぜ必要か**：部分構造の有無・個数を表す高次元特徴。方針3でブースティングに与える。

**修正5**：キャッシュmetadataに **SMILES列のSHA-1・データ種別(train/test)・includeChirality・
fingerprint種別(count/binary)・radius・fpSize・RDKitバージョン** を含める。行数が同じでも
SMILESが変われば SHA-1 が変わり、必ず再計算される。

In [10]:
def calc_morgan(mols, radius: int, nbits: int, count: bool, include_chirality: bool = False) -> sp.csr_matrix:
    """Morgan FP を疎行列(CSR, float32)で返す。count=True で出現回数、False で 0/1。"""
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nbits,
                                                    includeChirality=include_chirality)
    rows = []
    for m in tqdm(mols, desc=f"morgan(r{radius},{'cnt' if count else 'bin'})"):
        if m is None:
            rows.append(np.zeros(nbits, dtype=np.float32))
        elif count:
            rows.append(gen.GetCountFingerprintAsNumPy(m).astype(np.float32))
        else:
            rows.append(gen.GetFingerprintAsNumPy(m).astype(np.float32))
    return sp.csr_matrix(np.vstack(rows))

def _smiles_sha1(smiles_series) -> str:
    return hashlib.sha1("\n".join(pd.Series(smiles_series).astype(str)).encode("utf-8")).hexdigest()

def cached_morgan(data_kind: str, smiles_series, mols, radius, nbits, count,
                  include_chirality: bool = False) -> sp.csr_matrix:
    """Morgan FP を .npz にキャッシュ。metadataにSMILESのSHA-1等を含め、内容が変われば再計算する。
    data_kind: 'train' または 'test'。"""
    fp_kind = "count" if count else "binary"
    tag = f"{data_kind}_{fp_kind[0]}{radius}_{nbits}"
    npz, mj = CACHE_DIR / f"morgan_{tag}.npz", CACHE_DIR / f"morgan_{tag}.meta.json"
    meta = dict(data_kind=data_kind, smiles_sha1=_smiles_sha1(smiles_series),
                include_chirality=include_chirality, fingerprint_kind=fp_kind,
                radius=radius, fpSize=nbits, rdkit=rdkit.__version__, n=len(mols))
    if npz.exists() and mj.exists() and json.load(open(mj)) == meta:
        print(f"[cache hit] morgan_{tag}")
        return sp.load_npz(npz)
    M = calc_morgan(mols, radius, nbits, count, include_chirality)
    sp.save_npz(npz, M)
    json.dump(meta, open(mj, "w"), ensure_ascii=False, indent=2)
    print(f"[cache save] morgan_{tag}  shape={M.shape}")
    return M

# 第一候補（count, r2, 2048）を生成・キャッシュ
MFP_train = cached_morgan("train", train_df["smiles"], train_mols, 2, 2048, True)
MFP_test = cached_morgan("test", test_df["smiles"], test_mols, 2, 2048, True)
print("Morgan(count,r2,2048):", MFP_train.shape,
      "平均非ゼロ/分子:", round(MFP_train.nnz / MFP_train.shape[0], 2))

[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
Morgan(count,r2,2048): (15000, 2048) 平均非ゼロ/分子: 20.43


## 9. 特徴量クリーニング（修正3：CV前の中央値補完を削除）

**何をする処理か**：記述子＋独自特徴量を結合し、目的変数を使わない前処理（inf→NaN、定数列削除、完全重複列削除、
float32化）を行う。**中央値補完はしない**（LightGBM/XGBoostはNaNを扱える。方針2はPipeline内の`SimpleImputer`で補完）。

**なぜ必要か**：CV前に全trainの中央値で補完すると、検証foldの情報が学習に混じり得るリークになる。定数・重複列は
教師なしなので削除してよい。

**実行結果の読み方**：`train/test 列一致 = True`、inf が 0、残存NaN列数（これは方針1/3ではモデルが欠損として扱う）。

In [11]:
def clean_dense(train_feat: pd.DataFrame, test_feat: pd.DataFrame):
    """教師なし前処理：inf→NaN、定数列・完全重複列の削除、train/test列整合、float32化。
    中央値補完はしない（NaNは残す）。"""
    tr = train_feat.replace([np.inf, -np.inf], np.nan)
    te = test_feat.replace([np.inf, -np.inf], np.nan)
    # 定数列削除（NaNを除く一意数<=1、全NaN列も削除）
    nun = tr.nunique(dropna=True)
    keep = list(nun[nun > 1].index)
    dropped_const = [c for c in tr.columns if c not in set(keep)]
    tr = tr[keep]
    # 完全重複列削除（NaN位置も含めて一致する列。判定用にsentinel埋めして比較、実データはNaNのまま）
    SENT = np.float64(-9.87654321e17)
    probe = tr.fillna(SENT)
    keep2 = list(probe.T.drop_duplicates().index)
    dropped_dup = [c for c in tr.columns if c not in set(keep2)]
    tr = tr[keep2]
    te = te.reindex(columns=keep2)   # fill_value指定なし → 欠損はNaNのまま
    return tr.astype(np.float32), te.astype(np.float32), dropped_const, dropped_dup

# 方針1・2 で使う「記述子＋独自特徴量」
raw_train = pd.concat([desc_train, cust_train], axis=1)
raw_test = pd.concat([desc_test, cust_test], axis=1)
feat_dense, feat_dense_test, dropped_const, dropped_dup = clean_dense(raw_train, raw_test)
FEAT_NAMES = list(feat_dense.columns)

assert list(feat_dense.columns) == list(feat_dense_test.columns), "train/test 列不一致"
assert not np.isinf(feat_dense.to_numpy(np.float32)).any(), "train に inf 残存"
assert not np.isinf(feat_dense_test.to_numpy(np.float32)).any(), "test に inf 残存"
n_nan_tr = int(np.isnan(feat_dense.to_numpy(np.float32)).sum())
n_nan_te = int(np.isnan(feat_dense_test.to_numpy(np.float32)).sum())
print(f"結合前: {raw_train.shape[1]} 列 → 定数削除 {len(dropped_const)} / 完全重複削除 {len(dropped_dup)} → 採用 {len(FEAT_NAMES)} 列")
print("train/test 列一致:", list(feat_dense.columns) == list(feat_dense_test.columns))
print(f"残存NaN（補完せずモデルへ渡す）: train={n_nan_tr} 個 / test={n_nan_te} 個")
X_dense = feat_dense.to_numpy(np.float32)
X_dense_test = feat_dense_test.to_numpy(np.float32)

結合前: 270 列 → 定数削除 31 / 完全重複削除 22 → 採用 217 列
train/test 列一致: True
残存NaN（補完せずモデルへ渡す）: train=0 個 / test=0 個


## 10. ベースライン

**何をする処理か**：比較の下限として、(1) `gap_eV` 中央値予測、(2) 記述子＋Ridge、(3) 記述子＋RandomForest、
(4) 記述子＋LightGBM、(5) 記述子＋Morgan＋LightGBM を同一foldで評価する。Ridge/RandomForest は
`SimpleImputer→…` の Pipeline（fold内fitでリーク回避、NaN補完も内部）。LightGBM系は `fit_boost_with_refit`（修正4）を使う。

**なぜ必要か**：各方針が「単純な予測」や「素のブースティング」をどれだけ上回るかを測るため。RandomForest は §11 の
提出候補比較（修正11）にも使うため OOF/テスト予測を保存する。

In [12]:
def cv_sklearn(make_est: Callable[[], object], X_tr, X_te, y, fold_id):
    """共有foldで sklearn 推定器（毎foldで新規fit）をCV。oof, test_pred, fold_mae, time を返す。"""
    oof = np.zeros(len(y)); test_pred = np.zeros(X_te.shape[0]); fold_mae = []
    t0 = time.time()
    for tr, va in iter_folds(fold_id, N_SPLITS):
        est = make_est()
        est.fit(X_tr[tr], y[tr])
        oof[va] = est.predict(X_tr[va])
        test_pred += est.predict(X_te) / N_SPLITS
        fold_mae.append(mean_absolute_error(y[va], oof[va]))
    return oof, test_pred, fold_mae, time.time() - t0

# (1) 中央値予測
t0 = time.time(); oof_med = np.zeros(len(y)); mae_med = []
for tr, va in iter_folds(fold_id, N_SPLITS):
    med = np.median(y[tr]); oof_med[va] = med
    mae_med.append(mean_absolute_error(y[va], oof_med[va]))
register_result("baseline_median", "-", "median", 0, mae_med, time.time() - t0, {}, "-")

# (2) 記述子(+独自) + Ridge（Pipeline: 補完→標準化→Ridge）
oof, tp, fm, tt = cv_sklearn(
    lambda: Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler()), ("ridge", Ridge(alpha=10.0, random_state=SEED))]),
    X_dense, X_dense_test, y, fold_id)
register_result("baseline_ridge", "descriptors+custom", "Ridge", X_dense.shape[1], fm, tt, {"alpha": 10.0}, "-")

# (3) 記述子(+独自) + RandomForest（Pipeline: 補完→RF）。OOF/テスト予測を保存（提出候補比較用）
oof_rf, tp_rf, fm_rf, tt_rf = cv_sklearn(
    lambda: Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("rf", RandomForestRegressor(
                          n_estimators=(200 if QUICK else 600), n_jobs=-1, random_state=SEED,
                          criterion=("squared_error" if QUICK else "absolute_error")))]),
    X_dense, X_dense_test, y, fold_id)
register_result("baseline_rf", "descriptors+custom", "RandomForest", X_dense.shape[1], fm_rf, tt_rf, {},
                "-", oof=oof_rf, test_pred=tp_rf)

# (4) 記述子(+独自) + LightGBM（修正4：ES→学習fold全体で再学習）
t0 = time.time()
oof, tp, fm, adopt, bi = cv_boost(
    lambda f, tr, va: (X_dense[tr], X_dense[va], X_dense_test), y, fold_id, "lgb", None)
print("  LightGBM 採用木数:", adopt, " 内部best_iter:", bi)
register_result("baseline_lgbm_desc", "descriptors+custom", "LightGBM", X_dense.shape[1],
                fm, time.time() - t0, {"n_estimators_adopted": adopt}, "-", oof=oof, test_pred=tp)

# (5) 記述子(+独自) + Morgan + LightGBM（dense結合：記述子NaN=欠損、Morganゼロ=0）
Xc_tr = np.hstack([X_dense, MFP_train.toarray()]).astype(np.float32)
Xc_te = np.hstack([X_dense_test, MFP_test.toarray()]).astype(np.float32)
t0 = time.time()
oof, tp, fm, adopt, bi = cv_boost(
    lambda f, tr, va: (Xc_tr[tr], Xc_tr[va], Xc_te), y, fold_id, "lgb", None)
print("  LightGBM(+Morgan) 採用木数:", adopt, " 内部best_iter:", bi)
register_result("baseline_lgbm_desc_morgan", "descriptors+custom+morgan", "LightGBM",
                Xc_tr.shape[1], fm, time.time() - t0, {"n_estimators_adopted": adopt}, "-", oof=oof, test_pred=tp)

print("\n参考（既存NB qm9_gap_prediction.ipynb の自己出力・記録値）: 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098")
display(pd.DataFrame(RESULTS)[["strategy", "model", "n_features", "cv_mae_mean", "cv_mae_std"]])

[登録] baseline_median: CV MAE = 1.0751 ± 0.0078  (run_mode=FULL)
[登録] baseline_ridge: CV MAE = 0.3384 ± 0.0038  (run_mode=FULL)
[登録] baseline_rf: CV MAE = 0.2228 ± 0.0045  (run_mode=FULL)
  LightGBM 採用木数: [2000, 2000, 2000, 1998, 2000]  内部best_iter: [2000, 2000, 2000, 1998, 2000]
[登録] baseline_lgbm_desc: CV MAE = 0.2127 ± 0.0040  (run_mode=FULL)
  LightGBM(+Morgan) 採用木数: [2000, 1998, 2000, 1985, 2000]  内部best_iter: [2000, 1998, 2000, 1985, 2000]
[登録] baseline_lgbm_desc_morgan: CV MAE = 0.2093 ± 0.0037  (run_mode=FULL)

参考（既存NB qm9_gap_prediction.ipynb の自己出力・記録値）: 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098


,strategy,model,n_features,cv_mae_mean,cv_mae_std
0,baseline_median,median,0,1.075112,0.007771
1,baseline_ridge,Ridge,217,0.338386,0.003820
2,baseline_rf,RandomForest,217,0.222815,0.004518
3,baseline_lgbm_desc,LightGBM,217,0.212689,0.004049
4,baseline_lgbm_desc_morgan,LightGBM,2265,0.209326,0.003725


## 10.5 旧 MAE≈0.2098 モデルの再現（修正7）

**背景**：以前のノートブック（`qm9_gap_prediction.ipynb`）の「RDKit記述子＋Morgan＋LightGBM」が **CV MAE≈0.2098** と
記録されている。その条件を元ノートブックから確認して再実装し、今回の共通fold上で再評価する。

**元ノートブックから確認した条件**（`qm9_gap_prediction.ipynb` の実コードより）：
- 特徴量：`Descriptors.CalcMolDescriptors()` の **標準2D記述子のみ（独自特徴量なし）** ＋ Morgan **count / radius=2 / fpSize=2048**（`GetCountFingerprintAsNumPy`, includeChirality=False）。
- 前処理：inf→NaN → **train中央値で補完** → 定数列削除 → 記述子とMorganを別々にクリーニングして横結合。
- モデル：LightGBM `objective="regression_l1"`, `learning_rate=0.03`, `num_leaves=127`, `min_child_samples=20`,
  `subsample=0.8`, `subsample_freq=1`, `colsample_bytree=0.8`, `reg_lambda=1.0`, **`n_estimators=5000`**, `random_state=8`。
- fold：`KFold(n_splits=5, shuffle=True, random_state=8)`（**group非考慮**）。
- early stopping：**外側検証foldを `eval_set` にして** `early_stopping(100)`。予測は `best_iteration_` まで。

**注意**：この旧手順には 2つのリーク要因がある — (a) 通常KFoldなので**重複分子がfoldを跨ぐ**、
(b) **early stoppingを外側検証foldで行い**その fold を予測するため `best_iteration` が楽観的に選ばれる。
そこで下記の2通りを出す：
- **(A) 厳密再現**：旧コードをそのまま（旧KFold・検証foldでES）→ 0.2098 が再現するか検証。
- **(B) 誠実な再評価**：同じ特徴量・パラメータを**今回の共通GroupKFold＋`fit_boost_with_refit`**（リーク無し）で評価。
  これを `baseline_legacy_rdkit_morgan_lgbm` として登録する。

数値は捏造せず、実際の再実行結果を使う。

In [13]:
# 旧NBと同一の特徴量（標準記述子のみ + Morgan count r2/2048）と旧クリーニング（train中央値補完あり）
def legacy_clean(train_feat: pd.DataFrame, test_feat: pd.DataFrame):
    tr = train_feat.replace([np.inf, -np.inf], np.nan)
    te = test_feat.replace([np.inf, -np.inf], np.nan)
    med = tr.median(numeric_only=True)          # 旧処理：train中央値で補完（当時の条件を忠実再現）
    tr = tr.fillna(med); te = te.fillna(med)
    keep = list(tr.columns[tr.nunique() > 1])
    tr = tr[keep]; te = te.reindex(columns=keep, fill_value=0.0)
    return tr.astype(np.float32), te.astype(np.float32)

ldesc_tr, ldesc_te = legacy_clean(desc_train, desc_test)                       # 標準記述子のみ
_lmfp_tr = cached_morgan("train", train_df["smiles"], train_mols, 2, 2048, True)
_lmfp_te = cached_morgan("test", test_df["smiles"], test_mols, 2, 2048, True)
lmfp_tr, lmfp_te = legacy_clean(pd.DataFrame(_lmfp_tr.toarray()), pd.DataFrame(_lmfp_te.toarray()))
Xleg_tr = np.hstack([ldesc_tr.to_numpy(np.float32), lmfp_tr.to_numpy(np.float32)])
Xleg_te = np.hstack([ldesc_te.to_numpy(np.float32), lmfp_te.to_numpy(np.float32)])
print("旧再現の特徴量数:", Xleg_tr.shape[1], "（記述子", ldesc_tr.shape[1], "+ Morgan", lmfp_tr.shape[1], "）")

LEGACY_PARAMS = dict(objective="regression_l1", metric="mae", learning_rate=0.03, num_leaves=127,
                     min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                     reg_lambda=1.0, n_estimators=5000, random_state=SEED, n_jobs=-1, verbose=-1)

# (A) 厳密再現：旧KFold・検証foldでES（0.2098が再現するか）
def legacy_exact_cv(Xtr, Xte, y):
    kfL = KFold(n_splits=5, shuffle=True, random_state=8)
    oof = np.zeros(len(y)); tp = np.zeros(Xte.shape[0]); fm = []; bits = []
    for tr, va in kfL.split(Xtr):
        m = lgb.LGBMRegressor(**LEGACY_PARAMS)
        m.fit(Xtr[tr], y[tr], eval_set=[(Xtr[va], y[va])], eval_metric="mae",
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof[va] = m.predict(Xtr[va]); tp += m.predict(Xte) / 5
        fm.append(mean_absolute_error(y[va], oof[va])); bits.append(int(m.best_iteration_))
    return oof, tp, fm, bits

t0 = time.time()
oofLA, tpLA, fmLA, bitsLA = legacy_exact_cv(Xleg_tr, Xleg_te, y)
mae_exact = float(np.mean(fmLA))
print(f"(A) 厳密再現（旧KFold・検証foldでES・n_est=5000）: CV MAE = {mae_exact:.4f}  best_iters={bitsLA}")
print("    参考記録値 0.2098 との差:", round(mae_exact - 0.2098, 4))

# (B) 誠実な再評価：今回の共通GroupKFold + fit_boost_with_refit（リーク無し）
legacy_refit_params = dict(learning_rate=0.03, num_leaves=127, min_child_samples=20,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           reg_lambda=1.0, n_estimators=5000)
oof_leg, test_leg, fm_leg, adopt_leg, bi_leg = cv_boost(
    lambda f, tr, va: (Xleg_tr[tr], Xleg_tr[va], Xleg_te), y, fold_id, "lgb", legacy_refit_params)
mae_honest = float(np.mean(fm_leg))
print(f"(B) 誠実再評価（GroupKFold・再学習）: CV MAE = {mae_honest:.4f}  採用木数={adopt_leg}")

legacy_report = dict(exact_replica_cv_mae=mae_exact, exact_replica_best_iters=bitsLA,
                     honest_groupkfold_cv_mae=mae_honest, honest_adopted_estimators=adopt_leg,
                     recorded_value=0.2098,
                     cause_candidates=[
                         "fold分割の違い（旧KFoldはgroup非考慮で重複分子がfoldを跨ぐ→楽観化）",
                         "early stopping方法の違い（旧は外側検証foldでES→best_iterが楽観的）",
                         "本NBは学習fold全体で再学習しESリークを排除",
                         "RDKitバージョンの違い（記述子・FPの数値差）: " + rdkit.__version__,
                         "特徴量構成の違い（旧は独自特徴量なし・別々クリーニング）",
                     ])
json.dump(legacy_report, open(RESULTS_DIR / "legacy_reproduction.json", "w"), ensure_ascii=False, indent=2)

register_result("baseline_legacy_rdkit_morgan_lgbm", "rdkit_desc(only)+morgan(count,r2,2048)",
                "LightGBM", Xleg_tr.shape[1], fm_leg, time.time() - t0,
                {**LEGACY_PARAMS, "n_estimators_adopted": adopt_leg,
                 "eval_protocol": "common GroupKFold + refit (leak-free)"},
                "-", oof=oof_leg, test_pred=test_leg,
                extra={"exact_replica_cv_mae": mae_exact, "recorded_value": 0.2098})
print("\n[修正7 まとめ] 厳密再現=%.4f / 誠実再評価=%.4f / 記録値=0.2098" % (mae_exact, mae_honest))

[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
旧再現の特徴量数: 2235 （記述子 187 + Morgan 2048 ）
(A) 厳密再現（旧KFold・検証foldでES・n_est=5000）: CV MAE = 0.2098  best_iters=[4996, 4987, 4296, 5000, 3918]
    参考記録値 0.2098 との差: 0.0
(B) 誠実再評価（GroupKFold・再学習）: CV MAE = 0.2091  採用木数=[4999, 4995, 5000, 4995, 3915]
[登録] baseline_legacy_rdkit_morgan_lgbm: CV MAE = 0.2091 ± 0.0034  (run_mode=FULL)

[修正7 まとめ] 厳密再現=0.2098 / 誠実再評価=0.2091 / 記録値=0.2098


## 11. 方針1：重要RDKit特徴量 ＋ ブースティング

**何をする処理か**：記述子＋独自特徴量から**重要特徴量を選び**、LightGBM/XGBoost を Optuna で調整する。
手順：(1)(2) 定数・完全重複列削除（§9済）→(3) 欠損はモデルへ→(4) 全特徴量で初期モデル→
(5) 重要度算出（Permutation / SHAP / gain）→(6) 上位特徴量数を変えて CV MAE 比較→(7) 最良数採用→(8) HP探索。

**なぜ必要か**：寄与の大きい特徴に絞って過学習と学習時間を抑えるため。

**リーク上の注意**：重要度は**各学習fold内**（内部分割）で算出し、fold ごとに独自の上位特徴を選ぶ。
特徴量数 k は共有CVで選ぶため、その CV MAE はやや楽観側になり得る（§14で相対比較）。優先順位は Permutation ≈ SHAP ＞ gain。

**修正9**：SHAPサンプルは先頭行ではなく seed 固定のランダム抽出にする（先頭偏りを避ける）。
**修正4**：CVは `fit_boost_with_refit`（ES→学習fold全体で再学習）を使う。

In [14]:
name2idx = {n: i for i, n in enumerate(FEAT_NAMES)}

def combined_ranking(gain, perm, shap_imp, names):
    """Permutation≈SHAP＞gain の優先で特徴量を降順ソートし、名前リストとスコアを返す。"""
    def norm(x):
        r = ss.rankdata(np.asarray(x, float)); return r / r.max()
    primary = 0.5 * norm(perm) + 0.5 * norm(shap_imp)
    score = primary + 1e-6 * norm(gain)  # gain は微小なタイブレークのみ
    order = np.argsort(-score)
    return [names[i] for i in order], score

# --- fold ごとの重要度ランキング（初期モデル＝全特徴 LightGBM、内部分割でES）---
fold_rank: Dict[int, List[str]] = {}
fold_importance_tables = []
for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
    xi, xv, yi, yv = train_test_split(X_dense[tr], y[tr], test_size=0.15, random_state=SEED + f)
    m = make_lgb()
    m.fit(xi, yi, eval_set=[(xv, yv)], eval_metric="mae",
          callbacks=[lgb.early_stopping(CFG["early_stopping_rounds"], verbose=False), lgb.log_evaluation(0)])
    gain = m.booster_.feature_importance("gain")
    perm = permutation_importance(m, xv, yv, n_repeats=CFG["s1_perm_repeats"], random_state=SEED,
                                  scoring="neg_mean_absolute_error", n_jobs=-1).importances_mean
    # 修正9：SHAPサンプルは seed 固定のランダム抽出
    rng = np.random.RandomState(SEED + f)
    n_sh = min(CFG["s1_shap_sample"], xi.shape[0])
    sample_idx = rng.choice(xi.shape[0], size=n_sh, replace=False)
    sh_sample = xi[sample_idx]
    shap_imp = np.abs(shap.TreeExplainer(m).shap_values(sh_sample)).mean(axis=0)
    names, score = combined_ranking(gain, perm, shap_imp, FEAT_NAMES)
    fold_rank[f] = names
    fold_importance_tables.append(pd.DataFrame({"feature": FEAT_NAMES, "gain": gain, "perm": perm,
                                                "shap": shap_imp, "score": score, "fold": f}))
    print(f"  fold{f}: best_iter={m.best_iteration_}  top5={names[:5]}")

imp_all = pd.concat(fold_importance_tables)
rank_tbl = imp_all.copy()
rank_tbl["rank"] = rank_tbl.groupby("fold")["score"].rank(ascending=False)
agg_rank = rank_tbl.groupby("feature")["rank"].mean().sort_values()
agg_rank.to_csv(RESULTS_DIR / "s1_aggregated_importance.csv")
imp_all.to_csv(RESULTS_DIR / "s1_fold_importance.csv", index=False)
print("統合重要度 上位10:", list(agg_rank.index[:10]))

  fold0: best_iter=2000  top5=['n_sp2', 'n_conj_bond', 'FractionCSP3', 'BCUT2D_MRHI', 'fr_aldehyde']
  fold1: best_iter=2000  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde']
  fold2: best_iter=2000  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde']
  fold3: best_iter=1998  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde']
  fold4: best_iter=2000  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'HallKierAlpha', 'r_multiple']
統合重要度 上位10: ['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde', 'fr_ketone', 'BertzCT', 'HallKierAlpha', 'VSA_EState2', 'r_multiple']


In [15]:
def topk_idx(f: int, k) -> List[int]:
    names = fold_rank[f] if k == "all" else fold_rank[f][:k]
    return [name2idx[n] for n in names]

def s1_oof(k, params, kind="lgb"):
    """fold内 top-k 選択でブースティングをCV（fit_boost_with_refit）。oof,test_pred,fold_mae,採用木数 を返す。"""
    oof = np.zeros(len(y)); test_pred = np.zeros(X_dense_test.shape[0]); fm = []; adopt = []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        cols = topk_idx(f, k)
        Xtr, Xva, Xte = X_dense[tr][:, cols], X_dense[va][:, cols], X_dense_test[:, cols]
        model, n_ad, bi = fit_boost_with_refit(kind, params, Xtr, y[tr], SEED + f)
        oof[va] = model.predict(Xva)
        test_pred += model.predict(Xte) / N_SPLITS
        fm.append(mean_absolute_error(y[va], oof[va])); adopt.append(n_ad)
    return oof, test_pred, fm, adopt

# --- (6)(7) 特徴量数スイープ（固定LightGBMで比較）---
k_scores = {}
for k in CFG["s1_feature_counts"]:
    _, _, fm, _ = s1_oof(k, None, "lgb")
    k_scores[str(k)] = float(np.mean(fm))
    print(f"  k={k}: CV MAE = {k_scores[str(k)]:.4f}")
best_k_str = min(k_scores, key=k_scores.get)
best_k = "all" if best_k_str == "all" else int(best_k_str)
json.dump({"k_scores": k_scores, "best_k": best_k_str}, open(RESULTS_DIR / "s1_kscan.json", "w"), indent=2)
print("採用特徴量数 best_k =", best_k)

  k=10: CV MAE = 0.2762
  k=20: CV MAE = 0.2317
  k=40: CV MAE = 0.2194
  k=80: CV MAE = 0.2123
  k=120: CV MAE = 0.2140
  k=all: CV MAE = 0.2133
採用特徴量数 best_k = 80


In [16]:
# --- (8) Optuna ハイパーパラメータ探索（LightGBM / XGBoost）目的=5-fold CV平均MAE ---
def s1_obj_lgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 80),
        subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    return float(np.mean(s1_oof(best_k, p, "lgb")[2]))

def s1_obj_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    return float(np.mean(s1_oof(best_k, p, "xgb")[2]))

t0 = time.time()
study1_lgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                 study_name=f"s1_lgb_{RUN_MODE}", storage=OPTUNA_STORAGE, load_if_exists=True)
study1_lgb.optimize(s1_obj_lgb, n_trials=N_TRIALS_S1_LGB, show_progress_bar=True)
study1_lgb.trials_dataframe().to_csv(RESULTS_DIR / "strategy1_lgb_trials.csv", index=False)

study1_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                 study_name=f"s1_xgb_{RUN_MODE}", storage=OPTUNA_STORAGE, load_if_exists=True)
study1_xgb.optimize(s1_obj_xgb, n_trials=N_TRIALS_S1_XGB, show_progress_bar=True)
study1_xgb.trials_dataframe().to_csv(RESULTS_DIR / "strategy1_xgb_trials.csv", index=False)

best_kind, best_study = ("lgb", study1_lgb) if study1_lgb.best_value <= study1_xgb.best_value else ("xgb", study1_xgb)
print(f"方針1 最良モデル: {best_kind}  (LGB={study1_lgb.best_value:.4f} / XGB={study1_xgb.best_value:.4f})")

oof1, test1, fm1, adopt1 = s1_oof(best_k, best_study.best_params, best_kind)
assert np.isfinite(oof1).all(), "OOFに欠損/無限"
n_feat1 = len(FEAT_NAMES) if best_k == "all" else best_k
sub1 = make_submission(test1, "submission_strategy1.csv", test_df)
register_result("strategy1", f"desc+custom top{best_k}", best_kind.upper(), n_feat1, fm1, time.time() - t0,
                {**best_study.best_params, "best_k": best_k, "n_estimators_adopted": adopt1},
                "submission_strategy1.csv", oof=oof1, test_pred=test1)

Best trial: 77. Best value: 0.203619: 100%|██████████| 30/30 [1:10:32<00:00, 141.08s/it]


方針1 最良モデル: lgb  (LGB=0.2033 / XGB=0.2036)
保存: submission_strategy1.csv  (4000行)
[登録] strategy1: CV MAE = 0.2038 ± 0.0045  (run_mode=FULL)


{'strategy': 'strategy1',
 'feature_set': 'desc+custom top80',
 'model': 'LGB',
 'n_features': 80,
 'run_mode': 'FULL',
 'quick': False,
 'cv_mae_mean': 0.2038453259213561,
 'cv_mae_std': 0.004498796491033085,
 'fold1_mae': 0.20405818559352967,
 'fold2_mae': 0.2005848964742208,
 'fold3_mae': 0.2038773819292123,
 'fold4_mae': 0.19879669467227695,
 'fold5_mae': 0.2119094709375407,
 'training_time': 15758.169573783875,
 'best_params': '{"n_estimators": 1500, "learning_rate": 0.0233342321525001, "num_leaves": 243, "max_depth": 12, "min_child_samples": 12, "subsample": 0.8246134487278344, "colsample_bytree": 0.7269525532574, "reg_alpha": 0.0020952103873760606, "reg_lambda": 0.44271305184117504, "best_k": 80, "n_estimators_adopted": [1500, 1497, 1495, 1500, 1500]}',
 'submission_path': 'submission_strategy1.csv'}

## 12. 方針2：PCA ＋ RBF-SVR（修正8：全PCA候補を比較）

**何をする処理か**：「記述子＋独自特徴量」（Morganは使わない）に `SimpleImputer→StandardScaler→PCA→SVR(rbf)` の
**Pipeline** を各fold内でfitしてCVする。PCA設定は次の2段階で選ぶ：
- **第1段階（粗探索）**：全PCA候補（累積寄与率 0.90/0.95/0.97/0.99、主成分数 20/40/80/120）を、固定SVRで**最低1回ずつ**評価。
- **第2段階（詳細探索）**：成績上位2設定について Optuna で `C`/`gamma`/`epsilon` を探索。

**なぜ必要か**：PCA設定をOptunaのcategoricalに混ぜると一部設定が十分評価されない。全候補を必ず1回は比較する。

**リーク上の注意**：`SimpleImputer`/`StandardScaler`/`PCA` は Pipeline 内で**各fold学習データだけ**でfitする。
**修正8**：最終CVは全15,000件で評価する。FULLでは探索時もサブサンプルしない（`s2_svr_max_train=None`）。

In [17]:
min_fold_train = len(y) - int(np.bincount(fold_id).max())   # 最小の学習fold件数
max_nc = min(len(FEAT_NAMES), min_fold_train)
pca_settings = [("var", v) for v in CFG["s2_pca_var"]] + \
               [("nc", k) for k in CFG["s2_pca_ncomp"] if k <= max_nc]
print("PCA候補:", pca_settings, "| max_nc:", max_nc)

def build_pipe(pca_kind, pca_val, C, gamma, epsilon):
    n_comp = pca_val if pca_kind == "var" else int(pca_val)
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler()),
                     ("pca", PCA(n_components=n_comp, random_state=SEED)),
                     ("svr", SVR(kernel="rbf", C=C, gamma=gamma, epsilon=epsilon))])

def s2_cv(pca_kind, pca_val, C, gamma, epsilon, subsample=None):
    """Pipeline を各fold内でfitしてCV。oof,test_pred,fold_mae を返す。"""
    oof = np.zeros(len(y)); test_pred = np.zeros(X_dense_test.shape[0]); fm = []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        tr_use = tr
        if subsample is not None and len(tr) > subsample:
            rng = np.random.RandomState(SEED + f)
            tr_use = rng.choice(tr, size=subsample, replace=False)
        pipe = build_pipe(pca_kind, pca_val, C, gamma, epsilon)
        pipe.fit(X_dense[tr_use], y[tr_use])
        oof[va] = pipe.predict(X_dense[va])
        test_pred += pipe.predict(X_dense_test) / N_SPLITS
        fm.append(mean_absolute_error(y[va], oof[va]))
    return oof, test_pred, fm

# --- 第1段階：全PCA候補を固定SVRで1回ずつ評価 ---
cs = CFG["s2_coarse_svr"]
coarse = []
for pk, pv in pca_settings:
    _, _, fm = s2_cv(pk, pv, cs["C"], cs["gamma"], cs["epsilon"], subsample=CFG["s2_svr_max_train"])
    coarse.append(dict(pca=f"{pk}:{pv}", pca_kind=pk, pca_val=pv, cv_mae=float(np.mean(fm))))
    print(f"  PCA {pk}:{pv}  固定SVR CV MAE = {np.mean(fm):.4f}")
coarse_df = pd.DataFrame(coarse).sort_values("cv_mae").reset_index(drop=True)
coarse_df.to_csv(RESULTS_DIR / "s2_pca_coarse.csv", index=False)
top_pca = [(r.pca_kind, r.pca_val) for r in coarse_df.head(CFG["s2_top_pca"]).itertuples()]
print("上位PCA設定（詳細探索対象）:", top_pca)

PCA候補: [('var', 0.9), ('var', 0.95), ('var', 0.97), ('var', 0.99), ('nc', 20), ('nc', 40), ('nc', 80), ('nc', 120)] | max_nc: 217
  PCA var:0.9  固定SVR CV MAE = 0.2562
  PCA var:0.95  固定SVR CV MAE = 0.2490
  PCA var:0.97  固定SVR CV MAE = 0.2441
  PCA var:0.99  固定SVR CV MAE = 0.2312
  PCA nc:20  固定SVR CV MAE = 0.3034
  PCA nc:40  固定SVR CV MAE = 0.2647
  PCA nc:80  固定SVR CV MAE = 0.2443
  PCA nc:120  固定SVR CV MAE = 0.2248
上位PCA設定（詳細探索対象）: [('nc', 120.0), ('var', 0.99)]


In [18]:
# --- 第2段階：上位PCA設定について Optuna で C/gamma/epsilon を探索 ---
pca_choices = [f"{k}:{v}" for k, v in top_pca]
s2_timing = []

def s2_obj(trial):
    sel = trial.suggest_categorical("pca", pca_choices)
    pk, pv = sel.split(":"); pv = float(pv) if pk == "var" else int(float(pv))
    C = trial.suggest_float("C", 1e-1, 1e4, log=True)
    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "value"])
    gamma = "scale" if gamma_mode == "scale" else trial.suggest_float("gamma", 1e-5, 1e0, log=True)
    epsilon = trial.suggest_float("epsilon", 1e-3, 10 ** -0.3, log=True)
    t0 = time.time()
    _, _, fm = s2_cv(pk, pv, C, gamma, epsilon, subsample=CFG["s2_svr_max_train"])
    s2_timing.append(dict(trial=trial.number, pca=f"{pk}:{pv}", C=C, gamma=str(gamma),
                          epsilon=epsilon, mae=float(np.mean(fm)), total_time=round(time.time() - t0, 2)))
    return float(np.mean(fm))

t0 = time.time()
study2 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                             study_name=f"s2_svr_{RUN_MODE}", storage=OPTUNA_STORAGE, load_if_exists=True)
study2.optimize(s2_obj, n_trials=N_TRIALS_S2_SVR, show_progress_bar=True)
study2.trials_dataframe().to_csv(RESULTS_DIR / "strategy2_svr_trials.csv", index=False)
pd.DataFrame(s2_timing).to_csv(RESULTS_DIR / "s2_timing.csv", index=False)
print("方針2 best params:", study2.best_params, " best MAE:", round(study2.best_value, 4))

Best trial: 60. Best value: 0.222545: 100%|██████████| 40/40 [3:51:33<00:00, 347.34s/it]  

方針2 best params: {'pca': 'nc:120.0', 'C': 7.4684399659315295, 'gamma_mode': 'scale', 'epsilon': 0.023594837883752502}  best MAE: 0.2225


In [19]:
# --- 最終CV：全15,000件で評価（サブサンプルなし）→ 提出 ---
bp = study2.best_params
pk, pv = bp["pca"].split(":"); pv = float(pv) if pk == "var" else int(float(pv))
gamma = "scale" if bp["gamma_mode"] == "scale" else bp["gamma"]
oof2, test2, fm2 = s2_cv(pk, pv, bp["C"], gamma, bp["epsilon"], subsample=None)   # 最終は全件

# 採用主成分数・累積寄与率（fold0の学習データで前処理のみfitして確認）
pre = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                ("pca", PCA(n_components=(pv if pk == "var" else int(pv)), random_state=SEED))])
tr0 = next(iter_folds(fold_id, N_SPLITS))[0]
pre.fit(X_dense[tr0])
n_pca = int(pre.named_steps["pca"].n_components_)
cum_var = float(pre.named_steps["pca"].explained_variance_ratio_.sum())
print(f"採用主成分数={n_pca}  累積寄与率={cum_var:.3f}  最終CV MAE={np.mean(fm2):.4f}")
assert np.isfinite(oof2).all()
sub2 = make_submission(test2, "submission_strategy2.csv", test_df)
register_result("strategy2", "desc+custom+PCA", "RBF-SVR", n_pca, fm2, time.time() - t0,
                {**bp, "n_pca": n_pca, "cum_var": cum_var}, "submission_strategy2.csv", oof=oof2, test_pred=test2)

採用主成分数=120  累積寄与率=0.997  最終CV MAE=0.2225
保存: submission_strategy2.csv  (4000行)
[登録] strategy2: CV MAE = 0.2225 ± 0.0022  (run_mode=FULL)


{'strategy': 'strategy2',
 'feature_set': 'desc+custom+PCA',
 'model': 'RBF-SVR',
 'n_features': 120,
 'run_mode': 'FULL',
 'quick': False,
 'cv_mae_mean': 0.22254487888429755,
 'cv_mae_std': 0.0021897400770920297,
 'fold1_mae': 0.22316291893522036,
 'fold2_mae': 0.2215784674444274,
 'fold3_mae': 0.22073808174695939,
 'fold4_mae': 0.22070082047022777,
 'fold5_mae': 0.22654410582465284,
 'training_time': 14278.788572788239,
 'best_params': '{"pca": "nc:120.0", "C": 7.4684399659315295, "gamma_mode": "scale", "epsilon": 0.023594837883752502, "n_pca": 120, "cum_var": 0.9968916177749634}',
 'submission_path': 'submission_strategy2.csv'}

## 13. 方針3：大域RDKit ＋ 独自特徴量 ＋ Morgan ＋ ブースティング（修正6：モデル別に特徴量設定を選ぶ）

**何をする処理か**：`FpDensityMorgan1/2/3` は Morgan と直接重なるため常に除外。`fr_*`（フラグメント記述子）は
**除外/残すの両方を比較**。高相関列の削除しきい値、Morganの count/binary・radius・低頻度bit削除条件を CV MAE で比較する。
**修正6**：これらの設定選択を **LightGBM と XGBoost で別々**に行い、各モデルの最良構成同士を最後に比較する。

**なぜ必要か**：LightGBMで最適な特徴量設定がXGBoostでも最適とは限らないため、モデルごとに設定を選ぶ。

**リーク上の注意**：相関・低頻度bitの判定は教師なし（目的変数不使用）。相関計算のための欠損補完は判定用のみ（学習には使わない）。
過学習は列サンプリング（`colsample_bytree`）と正則化（`reg_alpha/lambda`）で抑える。

In [20]:
FPDENS = {"FpDensityMorgan1", "FpDensityMorgan2", "FpDensityMorgan3"}
_desc_block_cache: Dict[bool, tuple] = {}

def desc_block(include_fr: bool):
    """記述子(fr_*除外/残す) + 独自特徴量 を教師なしクリーニングして dense を返す（キャッシュ）。"""
    if include_fr in _desc_block_cache:
        return _desc_block_cache[include_fr]
    drop = [c for c in desc_train.columns if c in FPDENS or ((not include_fr) and c.startswith("fr_"))]
    gtr, gte, _, _ = clean_dense(pd.concat([desc_train.drop(columns=drop), cust_train], axis=1),
                                 pd.concat([desc_test.drop(columns=drop), cust_test], axis=1))
    res = (gtr.to_numpy(np.float32), gte.to_numpy(np.float32), list(gtr.columns))
    _desc_block_cache[include_fr] = res
    return res

def corr_keep_idx(X, thr):
    """|相関|>thr の冗長列を貪欲に削除して残す列indexを返す（thr=Noneで全列）。判定用にNaNは列中央値補完。"""
    if thr is None:
        return list(range(X.shape[1]))
    med = np.nanmedian(X, axis=0)
    Xi = np.where(np.isnan(X), med, X)
    c = np.nan_to_num(np.corrcoef(Xi, rowvar=False))
    keep, dropped = [], set()
    for i in range(X.shape[1]):
        if i in dropped:
            continue
        keep.append(i)
        for j in np.where(np.abs(c[i]) > thr)[0]:
            if j > i:
                dropped.add(j)
    return keep

def filter_lowfreq(M_tr, M_te, min_df):
    """train上で出現分子数>=min_df のbitのみ残す（教師なし）。"""
    dfc = np.asarray((M_tr > 0).sum(axis=0)).ravel()
    keep = np.where(dfc >= min_df)[0]
    return M_tr[:, keep].tocsr(), M_te[:, keep].tocsr(), keep

def assemble_dense(include_fr, corr_thr, morgan_kind, radius, nbits, min_df):
    """記述子(相関削除後) + Morgan(低頻度削除後) を dense 結合して返す（記述子NaN=欠損, Morganゼロ=0）。"""
    gtr, gte, _ = desc_block(include_fr)
    cols = corr_keep_idx(gtr, corr_thr)
    Dtr, Dte = gtr[:, cols], gte[:, cols]
    count = (morgan_kind == "count")
    Mtr = cached_morgan("train", train_df["smiles"], train_mols, radius, nbits, count)
    Mte = cached_morgan("test", test_df["smiles"], test_mols, radius, nbits, count)
    Mtr_f, Mte_f, keep = filter_lowfreq(Mtr, Mte, min_df)
    XCtr = np.hstack([Dtr, Mtr_f.toarray()]).astype(np.float32)
    XCte = np.hstack([Dte, Mte_f.toarray()]).astype(np.float32)
    cfg = dict(include_fragments=include_fr, corr_threshold=corr_thr, morgan_kind=morgan_kind,
               radius=radius, fp_size=nbits, min_df=min_df, n_desc=len(cols),
               n_bits=int(len(keep)), n_features=int(XCtr.shape[1]))
    return XCtr, XCte, cfg

def sweep_eval(Xtr, y, kind, params, n_splits_used):
    """設定比較用の軽量CV（先頭 n_splits_used fold・内部ESのみ、再学習なし）。平均MAEを返す。"""
    fm = []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        if f >= n_splits_used:
            break
        xi, xv, yi, yv = train_test_split(Xtr[tr], y[tr], test_size=0.15, random_state=SEED + f)
        if kind == "lgb":
            m = lgb.LGBMRegressor(**_lgb_params(params, SEED + f))
            m.fit(xi, yi, eval_set=[(xv, yv)], eval_metric="mae",
                  callbacks=[lgb.early_stopping(CFG["early_stopping_rounds"], verbose=False), lgb.log_evaluation(0)])
            pred = m.predict(Xtr[va])
        else:
            m = xgb.XGBRegressor(**_xgb_params(params, SEED + f), early_stopping_rounds=CFG["early_stopping_rounds"])
            m.fit(xi, yi, eval_set=[(xv, yv)], verbose=False)
            bi = m.best_iteration
            pred = m.predict(Xtr[va], iteration_range=(0, (bi + 1) if bi is not None else 0))
        fm.append(mean_absolute_error(y[va], pred))
    return float(np.mean(fm))

In [21]:
# --- モデル別（LightGBM / XGBoost）に特徴量設定を選ぶ ---
KINDS = ["lgb", "xgb"]
SWEEP_PARAMS = dict(n_estimators=(400 if QUICK else 1200))
ref_morgan = CFG["s3_morgan_variants"][0]      # 記述子設定を選ぶ間は固定する参照Morgan
ref_min_df = CFG["s3_lowfreq_min_df"][0]
s3_sweep_rows, model_best = [], {}

for kind in KINDS:
    # Stage A: 記述子設定（fr_* 除外/残す × 高相関しきい値）を選ぶ（参照Morgan固定）
    best_desc = None
    for include_fr in CFG["s3_fr_options"]:
        for thr in CFG["s3_corr_thresholds"]:
            XCtr, _, cfg = assemble_dense(include_fr, thr, ref_morgan[0], ref_morgan[1], ref_morgan[2], ref_min_df)
            mae = sweep_eval(XCtr, y, kind, SWEEP_PARAMS, CFG["sweep_n_splits"])
            s3_sweep_rows.append(dict(model_kind=kind, stage="desc", include_fragments=include_fr,
                                      corr_threshold=str(thr), morgan_kind=ref_morgan[0], radius=ref_morgan[1],
                                      fp_size=ref_morgan[2], min_df=ref_min_df, n_features=cfg["n_features"], cv_mae=mae))
            print(f"  [{kind}|desc] fr={include_fr} corr={thr} n={cfg['n_features']}  sweep MAE={mae:.4f}")
            if best_desc is None or mae < best_desc["cv_mae"]:
                best_desc = dict(include_fr=include_fr, corr_thr=thr, cv_mae=mae)
    # Stage B: Morgan設定（kind/radius/nbits × 低頻度min_df）を選ぶ（Stage Aの最良記述子設定で）
    best_full = None
    for (mk, rad, nb) in CFG["s3_morgan_variants"]:
        for mdf in CFG["s3_lowfreq_min_df"]:
            XCtr, _, cfg = assemble_dense(best_desc["include_fr"], best_desc["corr_thr"], mk, rad, nb, mdf)
            mae = sweep_eval(XCtr, y, kind, SWEEP_PARAMS, CFG["sweep_n_splits"])
            s3_sweep_rows.append(dict(model_kind=kind, stage="morgan", include_fragments=best_desc["include_fr"],
                                      corr_threshold=str(best_desc["corr_thr"]), morgan_kind=mk, radius=rad,
                                      fp_size=nb, min_df=mdf, n_features=cfg["n_features"], cv_mae=mae))
            print(f"  [{kind}|morgan] {mk} r{rad} {nb} min_df={mdf} n={cfg['n_features']}  sweep MAE={mae:.4f}")
            if best_full is None or mae < best_full["cv_mae"]:
                best_full = dict(include_fr=best_desc["include_fr"], corr_thr=best_desc["corr_thr"],
                                 morgan_kind=mk, radius=rad, fp_size=nb, min_df=mdf, cv_mae=mae)
    model_best[kind] = best_full
    print(f"  → {kind} 最良特徴量設定: {best_full}")

pd.DataFrame(s3_sweep_rows).to_csv(RESULTS_DIR / "s3_feature_sweeps.csv", index=False)

# 各モデルの最良設定で最終特徴行列を構築
XC_by_kind = {}
for kind in KINDS:
    bf = model_best[kind]
    XCtr, XCte, cfg = assemble_dense(bf["include_fr"], bf["corr_thr"], bf["morgan_kind"], bf["radius"], bf["fp_size"], bf["min_df"])
    XC_by_kind[kind] = (XCtr, XCte, cfg)
    print(f"{kind} 最終特徴量数: {cfg['n_features']}")

[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=False corr=0.95 n=2179  sweep MAE=0.2225
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=False corr=0.98 n=2185  sweep MAE=0.2225
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=False corr=0.995 n=2196  sweep MAE=0.2218
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=False corr=None n=2206  sweep MAE=0.2215
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=True corr=0.95 n=2231  sweep MAE=0.2153
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=True corr=0.98 n=2240  sweep MAE=0.2160
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=True corr=0.995 n=2251  sweep MAE=0.2155
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [lgb|desc] fr=True corr=None n=2262  sweep MAE=0.2155
[cache hit] morgan_train_c

morgan(r2,bin): 100%|██████████| 15000/15000 [00:00<00:00, 33133.85it/s]


[cache save] morgan_train_b2_2048  shape=(15000, 2048)


morgan(r2,bin): 100%|██████████| 4000/4000 [00:00<00:00, 41275.41it/s]

[cache save] morgan_test_b2_2048  shape=(4000, 2048)


  [lgb|morgan] binary r2 2048 min_df=1 n=2231  sweep MAE=0.2165
[cache hit] morgan_train_b2_2048
[cache hit] morgan_test_b2_2048
  [lgb|morgan] binary r2 2048 min_df=2 n=2231  sweep MAE=0.2165
[cache hit] morgan_train_b2_2048
[cache hit] morgan_test_b2_2048
  [lgb|morgan] binary r2 2048 min_df=5 n=2231  sweep MAE=0.2165
[cache hit] morgan_train_b2_2048
[cache hit] morgan_test_b2_2048
  [lgb|morgan] binary r2 2048 min_df=10 n=2231  sweep MAE=0.2165


morgan(r3,cnt): 100%|██████████| 15000/15000 [00:00<00:00, 20193.07it/s]


[cache save] morgan_train_c3_2048  shape=(15000, 2048)


morgan(r3,cnt): 100%|██████████| 4000/4000 [00:00<00:00, 28019.61it/s]


[cache save] morgan_test_c3_2048  shape=(4000, 2048)
  [lgb|morgan] count r3 2048 min_df=1 n=2231  sweep MAE=0.2178
[cache hit] morgan_train_c3_2048
[cache hit] morgan_test_c3_2048
  [lgb|morgan] count r3 2048 min_df=2 n=2231  sweep MAE=0.2178
[cache hit] morgan_train_c3_2048
[cache hit] morgan_test_c3_2048
  [lgb|morgan] count r3 2048 min_df=5 n=2231  sweep MAE=0.2178
[cache hit] morgan_train_c3_2048
[cache hit] morgan_test_c3_2048
  [lgb|morgan] count r3 2048 min_df=10 n=2231  sweep MAE=0.2178
  → lgb 最良特徴量設定: {'include_fr': True, 'corr_thr': 0.95, 'morgan_kind': 'count', 'radius': 2, 'fp_size': 2048, 'min_df': 1, 'cv_mae': 0.2152742647426426}
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [xgb|desc] fr=False corr=0.95 n=2179  sweep MAE=0.2291
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [xgb|desc] fr=False corr=0.98 n=2185  sweep MAE=0.2289
[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  [xgb|desc] fr=False corr=0.995 n=21

In [22]:
# --- Optuna（LightGBM/XGBoost）を各モデルの最良特徴量設定で調整 ---
def s3_cv(kind, params):
    XCtr, XCte, _ = XC_by_kind[kind]
    return cv_boost(lambda f, tr, va: (XCtr[tr], XCtr[va], XCte), y, fold_id, kind, params)

def s3_obj_lgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.2, 0.9),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
    )
    return float(np.mean(s3_cv("lgb", p)[2]))

def s3_obj_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 30.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.2, 0.9),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
    )
    return float(np.mean(s3_cv("xgb", p)[2]))

t0 = time.time()
study3_lgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                 study_name=f"s3_lgb_{RUN_MODE}", storage=OPTUNA_STORAGE, load_if_exists=True)
study3_lgb.optimize(s3_obj_lgb, n_trials=N_TRIALS_S3_LGB, show_progress_bar=True)
study3_lgb.trials_dataframe().to_csv(RESULTS_DIR / "strategy3_lgb_trials.csv", index=False)

study3_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                 study_name=f"s3_xgb_{RUN_MODE}", storage=OPTUNA_STORAGE, load_if_exists=True)
study3_xgb.optimize(s3_obj_xgb, n_trials=N_TRIALS_S3_XGB, show_progress_bar=True)
study3_xgb.trials_dataframe().to_csv(RESULTS_DIR / "strategy3_xgb_trials.csv", index=False)

# 各モデルの最良構成（特徴量設定＋Optuna最良値）を保存
cfg_rows = []
for kind in KINDS:
    _, _, cfg = XC_by_kind[kind]
    cfg_rows.append(dict(model_kind=kind, corr_threshold=str(cfg["corr_threshold"]),
                         include_fragments=cfg["include_fragments"], morgan_kind=cfg["morgan_kind"],
                         radius=cfg["radius"], fp_size=cfg["fp_size"], min_df=cfg["min_df"],
                         n_features=cfg["n_features"],
                         cv_mae=(study3_lgb.best_value if kind == "lgb" else study3_xgb.best_value)))
s3_cfg_df = pd.DataFrame(cfg_rows)
s3_cfg_df.to_csv(RESULTS_DIR / "s3_model_configs.csv", index=False)
print("方針3 モデル別最良構成:"); display(s3_cfg_df)

best3_kind = "lgb" if study3_lgb.best_value <= study3_xgb.best_value else "xgb"
best3_study = study3_lgb if best3_kind == "lgb" else study3_xgb
print(f"方針3 最良モデル: {best3_kind}  (LGB={study3_lgb.best_value:.4f} / XGB={study3_xgb.best_value:.4f})")

oof3, test3, fm3, adopt3, bi3 = s3_cv(best3_kind, best3_study.best_params)
assert np.isfinite(oof3).all()
final_cfg = XC_by_kind[best3_kind][2]
sub3 = make_submission(test3, "submission_strategy3.csv", test_df)
register_result("strategy3", f"desc+custom+morgan[{best3_kind}]", best3_kind.upper(),
                final_cfg["n_features"], fm3, time.time() - t0,
                {**best3_study.best_params, "feature_config": final_cfg, "n_estimators_adopted": adopt3},
                "submission_strategy3.csv", oof=oof3, test_pred=test3)

Best trial: 9. Best value: 0.203729: 100%|██████████| 30/30 [2:26:13<00:00, 292.46s/it]  

方針3 モデル別最良構成:


,model_kind,corr_threshold,include_fragments,morgan_kind,radius,fp_size,min_df,n_features,cv_mae
0,lgb,0.95,True,count,2,2048,1,2231,0.204408
1,xgb,None,True,count,2,2048,1,2262,0.203729


方針3 最良モデル: xgb  (LGB=0.2044 / XGB=0.2037)
保存: submission_strategy3.csv  (4000行)
[登録] strategy3: CV MAE = 0.2037 ± 0.0046  (run_mode=FULL)


{'strategy': 'strategy3',
 'feature_set': 'desc+custom+morgan[xgb]',
 'model': 'XGB',
 'n_features': 2262,
 'run_mode': 'FULL',
 'quick': False,
 'cv_mae_mean': 0.20372856909652928,
 'cv_mae_std': 0.0045739619460743895,
 'fold1_mae': 0.2028012681234163,
 'fold2_mae': 0.20108029038797878,
 'fold3_mae': 0.20338763656031636,
 'fold4_mae': 0.1990179946019378,
 'fold5_mae': 0.2123556558089972,
 'training_time': 13215.04530096054,
 'best_params': '{"n_estimators": 1200, "learning_rate": 0.03358386146697082, "max_depth": 11, "min_child_weight": 18.02678018582362, "subsample": 0.8532577717027281, "colsample_bytree": 0.5574075726775277, "reg_alpha": 3.7163798969899977, "reg_lambda": 1.2579336784994701, "feature_config": {"include_fragments": true, "corr_threshold": null, "morgan_kind": "count", "radius": 2, "fp_size": 2048, "min_df": 1, "n_desc": 214, "n_bits": 2048, "n_features": 2262}, "n_estimators_adopted": [1200, 1200, 1200, 1199, 1197]}',
 'submission_path': 'submission_strategy3.csv'}

## 14. 3方針比較

**何をする処理か**：ベースラインと3方針を、strategy / model / n_features / cv_mae_mean / cv_mae_std / fold別MAE /
run_mode / training_time / best_params / submission_path の形式で一覧化し CSV 保存する。

**実行結果の読み方**：`cv_mae_mean` が小さいほど良い。`cv_mae_std` はfold間ばらつき。差が std に対して小さければ僅差。
`run_mode` が `FULL`（`quick=False`）であることを最終実行では確認する。

In [23]:
cmp = pd.DataFrame(RESULTS)
cols = ["strategy", "feature_set", "model", "n_features", "run_mode", "quick", "cv_mae_mean", "cv_mae_std",
        "fold1_mae", "fold2_mae", "fold3_mae", "fold4_mae", "fold5_mae",
        "training_time", "best_params", "submission_path"]
cmp = cmp[[c for c in cols if c in cmp.columns]].sort_values("cv_mae_mean").reset_index(drop=True)
cmp.to_csv(RESULTS_DIR / "strategy_comparison.csv", index=False)
display(cmp.drop(columns=["best_params"]))
best_row = cmp.iloc[0]
print(f"\n最良（全登録モデル中）: {best_row['strategy']}  CV MAE = {best_row['cv_mae_mean']:.4f} ± {best_row['cv_mae_std']:.4f}")
print(f"run_mode = {RUN_MODE} / QUICK = {QUICK}")

,strategy,feature_set,model,n_features,run_mode,quick,cv_mae_mean,cv_mae_std,fold1_mae,fold2_mae,fold3_mae,fold4_mae,fold5_mae,training_time,submission_path
0,strategy3,desc+custom+morgan[xgb],XGB,2262,FULL,False,0.203729,0.004574,0.202801,0.201080,0.203388,0.199018,0.212356,13215.045301,submission_strategy3.csv
1,strategy1,desc+custom top80,LGB,80,FULL,False,0.203845,0.004499,0.204058,0.200585,0.203877,0.198797,0.211909,15758.169574,submission_strategy1.csv
2,baseline_legacy_rdkit_morgan_lgbm,"rdkit_desc(only)+morgan(count,r2,2048)",LightGBM,2235,FULL,False,0.209098,0.003388,0.208301,0.207192,0.209114,0.205466,0.215418,2251.137566,-
3,baseline_lgbm_desc_morgan,descriptors+custom+morgan,LightGBM,2265,FULL,False,0.209326,0.003725,0.209911,0.207168,0.209373,0.204467,0.215712,191.142881,-
4,baseline_lgbm_desc,descriptors+custom,LightGBM,217,FULL,False,0.212689,0.004049,0.209940,0.209432,0.214125,0.209908,0.220039,132.094886,-
5,strategy2,desc+custom+PCA,RBF-SVR,120,FULL,False,0.222545,0.002190,0.223163,0.221578,0.220738,0.220701,0.226544,14278.788573,submission_strategy2.csv
6,baseline_rf,descriptors+custom,RandomForest,217,FULL,False,0.222815,0.004518,0.225649,0.219267,0.219725,0.219030,0.230403,1859.103023,-
7,baseline_ridge,descriptors+custom,Ridge,217,FULL,False,0.338386,0.003820,0.337472,0.337499,0.339709,0.332732,0.344519,0.725973,-
8,baseline_median,-,median,0,FULL,False,1.075112,0.007771,1.062762,1.074786,1.079844,1.072178,1.085991,0.004328,-



最良（全登録モデル中）: strategy3  CV MAE = 0.2037 ± 0.0046
run_mode = FULL / QUICK = False


## 15. ブレンド参考実験（正式方針ではない・参考候補）

**何をする処理か**：方針1・2・3 の OOF 予測を `重み>=0`・`合計=1` で組み合わせ、MAE を確認する。
同じOOFで重み最適化＆評価は楽観的になるため、**(a) 全OOF最適化（楽観・参考値）** と
**(b) fold外し推定（各foldは他foldで重み推定）** の両方を出す。提出候補としては (b) を用いる。

**なぜ必要か**：ブレンドは提出候補の1つとして参考比較する（**方針3を置き換えない**）。
レギュレーションの「モデル数」の解釈が曖昧なため、ブレンドは**参考候補**として明記する。

In [24]:
oofs = {"s1": oof1, "s2": oof2, "s3": oof3}
tests = {"s1": test1, "s2": test2, "s3": test3}

def opt_weights(oof_list, y):
    k = len(oof_list); P = np.vstack(oof_list).T
    cons = ({"type": "eq", "fun": lambda w: w.sum() - 1.0},)
    res = minimize(lambda w: mean_absolute_error(y, P @ w), np.full(k, 1.0 / k),
                   method="SLSQP", bounds=[(0.0, 1.0)] * k, constraints=cons)
    return res.x

combos = [("1+2", ["s1", "s2"]), ("1+3", ["s1", "s3"]), ("2+3", ["s2", "s3"]),
          ("1+2+3", ["s1", "s2", "s3"])]
rows = []
blend_oof = {}; blend_test = {}
for name, keys in combos:
    ol = [oofs[k] for k in keys]; tl = [tests[k] for k in keys]
    w_opt = opt_weights(ol, y)
    mae_opt = mean_absolute_error(y, np.vstack(ol).T @ w_opt)
    oof_nested = np.zeros(len(y))
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        w = opt_weights([o[tr] for o in ol], y[tr])
        oof_nested[va] = np.vstack([o[va] for o in ol]).T @ w
    mae_nested = mean_absolute_error(y, oof_nested)
    rows.append(dict(combo=name, weights=np.round(w_opt, 3).tolist(),
                     mae_oof_optimistic=round(mae_opt, 4), mae_nested=round(mae_nested, 4)))
    blend_oof[name] = oof_nested
    blend_test[name] = np.vstack(tl).T @ w_opt   # 提出用は全OOF最適重みでtestに適用
blend_df = pd.DataFrame(rows)
blend_df.to_csv(RESULTS_DIR / "blend_reference.csv", index=False)
print("※ mae_nested がより信頼できる参考値（mae_oof_optimistic は楽観側）")
display(blend_df)

# 参考候補として、最良の2モデルブレンドと3モデルブレンドの submission を出力（参考）
best2 = min([r for r in rows if r["combo"] != "1+2+3"], key=lambda r: r["mae_nested"])
sub_b2 = make_submission(blend_test[best2["combo"]], "submission_blend2_reference.csv", test_df)
sub_b3 = make_submission(blend_test["1+2+3"], "submission_blend3_reference.csv", test_df)
# ブレンドも候補表へ（nested MAE を CV 値として、参考フラグ付き）
register_result(f"blend_2model({best2['combo']})", "OOF blend", "blend(reference)", "-",
                [blend_df.set_index("combo").loc[best2["combo"], "mae_nested"]] * N_SPLITS, 0.0,
                {"weights": best2["weights"], "note": "reference only"}, "submission_blend2_reference.csv",
                oof=blend_oof[best2["combo"]], test_pred=blend_test[best2["combo"]],
                extra={"is_reference_blend": True})
register_result("blend_3model(1+2+3)", "OOF blend", "blend(reference)", "-",
                [blend_df.set_index("combo").loc["1+2+3", "mae_nested"]] * N_SPLITS, 0.0,
                {"weights": rows[-1]["weights"], "note": "reference only"}, "submission_blend3_reference.csv",
                oof=blend_oof["1+2+3"], test_pred=blend_test["1+2+3"], extra={"is_reference_blend": True})

※ mae_nested がより信頼できる参考値（mae_oof_optimistic は楽観側）


,combo,weights,mae_oof_optimistic,mae_nested
0,1+2,"[0.672, 0.328]",0.1974,0.1974
1,1+3,"[0.5, 0.5]",0.2014,0.2014
2,2+3,"[0.323, 0.677]",0.1978,0.1978
3,1+2+3,"[0.356, 0.296, 0.348]",0.1965,0.1965


保存: submission_blend2_reference.csv  (4000行)
保存: submission_blend3_reference.csv  (4000行)
[登録] blend_2model(1+2): CV MAE = 0.1974 ± 0.0000  (run_mode=FULL)
[登録] blend_3model(1+2+3): CV MAE = 0.1965 ± 0.0000  (run_mode=FULL)


{'strategy': 'blend_3model(1+2+3)',
 'feature_set': 'OOF blend',
 'model': 'blend(reference)',
 'n_features': '-',
 'run_mode': 'FULL',
 'quick': False,
 'cv_mae_mean': 0.1965,
 'cv_mae_std': 0.0,
 'fold1_mae': 0.1965,
 'fold2_mae': 0.1965,
 'fold3_mae': 0.1965,
 'fold4_mae': 0.1965,
 'fold5_mae': 0.1965,
 'training_time': 0.0,
 'best_params': '{"weights": [0.356, 0.296, 0.348], "note": "reference only"}',
 'submission_path': 'submission_blend3_reference.csv',
 'is_reference_blend': True}

## 16. 最終提出候補の選定（修正11：CV MAEとモデル多様性で決める）

**何をする処理か**：3方針の submission は必ず生成済み。そのうえで、旧ベースライン再現・RandomForest・
2/3モデルブレンドも**提出候補**として CV MAE で比較する。「方針1〜3だから自動的にこの3本」とはせず、
**CV MAE とモデル多様性**を根拠に最終候補3つを提示する。

**注意**：レギュレーションの「モデル数」の解釈が不明なため、ブレンドは**参考候補**として明記する
（単一モデルで3本に収める安全策と、ブレンドを含める攻めの策の両方を提示）。

In [25]:
cand = pd.DataFrame(RESULTS)
cand["is_reference_blend"] = cand.get("is_reference_blend", False)
# object dtype（NaN混在）だと ~ が bitwise not になるため、明示的に bool へ揃える
cand["is_reference_blend"] = cand["is_reference_blend"].fillna(False).astype(bool)
# 提出候補（中央値予測など自明ベースラインは除外）
exclude = {"baseline_median", "baseline_ridge"}
pool = cand[~cand["strategy"].isin(exclude)].copy().sort_values("cv_mae_mean").reset_index(drop=True)
print("=== 提出候補（CV MAE 昇順）===")
display(pool[["strategy", "model", "n_features", "cv_mae_mean", "cv_mae_std", "is_reference_blend", "submission_path"]])

# 多様性を考慮した単一モデル上位3（同一modelファミリに偏らないよう選ぶ）
singles = pool[~pool["is_reference_blend"]].copy()
picked, seen_models = [], set()
for _, r in singles.iterrows():
    fam = str(r["model"])
    if fam not in seen_models or len(picked) < 3:
        picked.append(r); seen_models.add(fam)
    if len(picked) >= 3:
        break
picked_df = pd.DataFrame(picked)
print("\n【推奨・安全策】単一モデルの提出候補3つ（CV MAE＋モデル多様性）:")
for _, r in picked_df.iterrows():
    print(f"  - {r['strategy']:34s} {r['model']:14s} CV MAE={r['cv_mae_mean']:.4f}  → {r['submission_path']}")

best_blend = pool[pool["is_reference_blend"]].sort_values("cv_mae_mean").head(1)
if len(best_blend):
    b = best_blend.iloc[0]
    print(f"\n【参考・攻めの策】最良ブレンド候補: {b['strategy']}  CV MAE(nested)={b['cv_mae_mean']:.4f}  → {b['submission_path']}")
print("\n必ず生成される正式3方針: submission_strategy1.csv / submission_strategy2.csv / submission_strategy3.csv")

final_selection = dict(run_mode=RUN_MODE, quick=QUICK,
                       recommended_single_models=[
                           dict(strategy=r["strategy"], model=r["model"],
                                cv_mae=float(r["cv_mae_mean"]), submission=r["submission_path"])
                           for _, r in picked_df.iterrows()],
                       reference_best_blend=(None if not len(best_blend) else
                           dict(strategy=str(best_blend.iloc[0]["strategy"]),
                                cv_mae=float(best_blend.iloc[0]["cv_mae_mean"]),
                                submission=str(best_blend.iloc[0]["submission_path"]))),
                       official_strategy_submissions=["submission_strategy1.csv", "submission_strategy2.csv", "submission_strategy3.csv"])
json.dump(final_selection, open(RESULTS_DIR / "final_selection.json", "w"), ensure_ascii=False, indent=2)

=== 提出候補（CV MAE 昇順）===


,strategy,model,n_features,cv_mae_mean,cv_mae_std,is_reference_blend,submission_path
0,blend_3model(1+2+3),blend(reference),-,0.196500,0.000000,True,submission_blend3_reference.csv
1,blend_2model(1+2),blend(reference),-,0.197400,0.000000,True,submission_blend2_reference.csv
2,strategy3,XGB,2262,0.203729,0.004574,False,submission_strategy3.csv
3,strategy1,LGB,80,0.203845,0.004499,False,submission_strategy1.csv
4,baseline_legacy_rdkit_morgan_lgbm,LightGBM,2235,0.209098,0.003388,False,-
5,baseline_lgbm_desc_morgan,LightGBM,2265,0.209326,0.003725,False,-
6,baseline_lgbm_desc,LightGBM,217,0.212689,0.004049,False,-
7,strategy2,RBF-SVR,120,0.222545,0.002190,False,submission_strategy2.csv
8,baseline_rf,RandomForest,217,0.222815,0.004518,False,-



【推奨・安全策】単一モデルの提出候補3つ（CV MAE＋モデル多様性）:
  - strategy3                          XGB            CV MAE=0.2037  → submission_strategy3.csv
  - strategy1                          LGB            CV MAE=0.2038  → submission_strategy1.csv
  - baseline_legacy_rdkit_morgan_lgbm  LightGBM       CV MAE=0.2091  → -

【参考・攻めの策】最良ブレンド候補: blend_3model(1+2+3)  CV MAE(nested)=0.1965  → submission_blend3_reference.csv

必ず生成される正式3方針: submission_strategy1.csv / submission_strategy2.csv / submission_strategy3.csv


## 17. 提出CSV生成と検証

**何をする処理か**：3方針の提出CSV（＋参考ブレンド）を再読込し、行数・列順・SMILES順・欠損/無限/重複を最終検証する。

**実行結果の読み方**：全ファイルで `OK` が出れば提出フォーマット要件（`smiles,gap_eV`・4000行）を満たす。

In [26]:
check_paths = ["submission_strategy1.csv", "submission_strategy2.csv", "submission_strategy3.csv",
               "submission_blend2_reference.csv", "submission_blend3_reference.csv"]
for path in check_paths:
    if not Path(path).exists():
        print(f"skip (not found): {path}"); continue
    s = pd.read_csv(path)
    assert list(s.columns) == ["smiles", "gap_eV"], f"{path}: 列順不正"
    assert len(s) == len(test_df), f"{path}: 行数不一致"
    assert s["smiles"].tolist() == test_df["smiles"].tolist(), f"{path}: SMILES順不一致"
    assert s["gap_eV"].notna().all() and np.isfinite(s["gap_eV"]).all(), f"{path}: 欠損/無限"
    print(f"OK  {path}  ({len(s)}行)  mean={s['gap_eV'].mean():.3f}")
print("\n（正式提出は strategy1/2/3、blend系は参考候補）")

OK  submission_strategy1.csv  (4000行)  mean=6.880
OK  submission_strategy2.csv  (4000行)  mean=6.875
OK  submission_strategy3.csv  (4000行)  mean=6.885
OK  submission_blend2_reference.csv  (4000行)  mean=6.879
OK  submission_blend3_reference.csv  (4000行)  mean=6.880

（正式提出は strategy1/2/3、blend系は参考候補）


## 18. 最終結果まとめ

**何をする処理か**：比較表から各モデルのCV MAEと最良を要約表示し、旧0.2098の再現状況と最終候補を確認する。

**用語メモ**：QM9＝最大9重原子(C,N,O,F)の有機分子の量子化学データ／RDKit＝ケモインフォマティクスライブラリ／
SMILES＝分子構造の文字列表記／分子記述子＝分子を数値化した量／フィンガープリント＝部分構造をビット列で表す特徴。

In [27]:
print(f"=== 最終結果まとめ  run_mode={RUN_MODE} / QUICK={QUICK} ===\n")
print("各モデル CV MAE（小さいほど良い）:")
for _, r in pd.DataFrame(RESULTS).sort_values("cv_mae_mean").iterrows():
    print(f"  {r['strategy']:34s} {str(r['model']):14s} n={str(r['n_features']):>5} "
          f"MAE={r['cv_mae_mean']:.4f} ± {r['cv_mae_std']:.4f}")
best = pd.DataFrame(RESULTS).sort_values("cv_mae_mean").iloc[0]
print(f"\n最良（全登録）: {best['strategy']}  ({best['model']})  CV MAE={best['cv_mae_mean']:.4f}")
print("\n[旧0.2098モデルの再現]")
print(f"  厳密再現(旧KFold・検証foldでES) = {legacy_report['exact_replica_cv_mae']:.4f}")
print(f"  誠実再評価(GroupKFold・再学習)  = {legacy_report['honest_groupkfold_cv_mae']:.4f}")
print(f"  記録値                         = 0.2098")
print("  原因候補:", *["\n    - " + c for c in legacy_report["cause_candidates"]])
print("\n参考: 既存NBベースライン 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098")

=== 最終結果まとめ  run_mode=FULL / QUICK=False ===

各モデル CV MAE（小さいほど良い）:
  blend_3model(1+2+3)                blend(reference) n=    - MAE=0.1965 ± 0.0000
  blend_2model(1+2)                  blend(reference) n=    - MAE=0.1974 ± 0.0000
  strategy3                          XGB            n= 2262 MAE=0.2037 ± 0.0046
  strategy1                          LGB            n=   80 MAE=0.2038 ± 0.0045
  baseline_legacy_rdkit_morgan_lgbm  LightGBM       n= 2235 MAE=0.2091 ± 0.0034
  baseline_lgbm_desc_morgan          LightGBM       n= 2265 MAE=0.2093 ± 0.0037
  baseline_lgbm_desc                 LightGBM       n=  217 MAE=0.2127 ± 0.0040
  strategy2                          RBF-SVR        n=  120 MAE=0.2225 ± 0.0022
  baseline_rf                        RandomForest   n=  217 MAE=0.2228 ± 0.0045
  baseline_ridge                     Ridge          n=  217 MAE=0.3384 ± 0.0038
  baseline_median                    median         n=    0 MAE=1.0751 ± 0.0078

最良（全登録）: blend_3model(1+2+3)  (blend(reference)

## 19. 再現方法

**環境構築**
```bash
cd lesson_9/self-code
uv sync
```

**実行**
- 動作確認（軽量）：§2の設定セルで `QUICK = True` にして上から全セル実行。
- フル探索（最終）：`QUICK = False` のまま全セル実行。または CLI で:
```bash
uv run jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 qm9_gap_prediction_final.ipynb
```
CPUのみ・GPU不要。FULL は数時間〜一晩が目安。短縮するには §2 の `N_TRIALS_*` や `boost_nest_range` を小さくする。

**再現性**
- seed=8 を Python/NumPy/scikit-learn/LightGBM/XGBoost/Optuna(TPESampler) に統一。
- fold は canonical SMILES グループの `GroupKFold`（§5, `results/fold_assignment.csv` / `results/fold_id.npy`）で全方針共有。
- 特徴量は `cache/`（metaにSMILESハッシュ等）に、OOF/テスト予測・trial履歴・結果JSONは `results/` に保存。
- Optuna study は SQLite（`results/optuna.db`）に保存し中断後に再開可能。

**ライブラリバージョン**（この環境）:

In [28]:
print(json.dumps(LIB_VERSIONS, ensure_ascii=False, indent=2))
json.dump(LIB_VERSIONS, open(RESULTS_DIR / "lib_versions.json", "w"), ensure_ascii=False, indent=2)

{
  "python": "3.11.15",
  "platform": "macOS-14.5-arm64-arm-64bit",
  "machine": "arm64",
  "numpy": "2.4.6",
  "pandas": "3.0.3",
  "scipy": "1.17.1",
  "scikit-learn": "1.9.0",
  "lightgbm": "4.7.0",
  "xgboost": "3.2.0",
  "optuna": "4.9.0",
  "shap": "0.51.0",
  "rdkit": "2026.03.4"
}
